# Tugas Kelompok 2: Pre-Processing per Research Question (RQ1, RQ2, RQ3) pada Dataset Gabungan X dan YouTube

## Kelompok

1. Erland Hafizh Aristovi (2306165686)
2. Jasmine Rakhaila (2306165774)
3. Leonita Cecilia (2306165710)
4. Yeshua Marco Gracia Manurung (2306210286)

## Gambaran Umum

**Topik (TK-1):** Analisis kekhawatiran publik terhadap kebersihan dan keamanan pangan Program Makan Bergizi Gratis (MBG) pada YouTube dan X.

Notebook ini adalah **tahap pre-processing lanjutan** setelah data X dan YouTube digabung (`data/processed/raw_merged/`). Cakupannya sampai **persiapan data** untuk tiga research question; **pemodelan (topic modelling, analisis lokasi, dan klasifikasi sentimen) belum dikerjakan** dan akan dilakukan pada tahap berikutnya.

| Pertanyaan penelitian (TK-1) | Metode nanti | Yang disiapkan di notebook ini | Kolom |
|---|---|---|---|
| **RQ1:** jenis permasalahan keamanan/kebersihan makanan | Topic modelling | Teks ringan (untuk model embedding, mis. BERTopic) dan token ter-stem tanpa stopword (untuk LDA/NMF), kategori isu sebagai label lemah | `rq1_*` |
| **RQ2:** variasi permasalahan antarwilayah | Analisis lokasi + label topik RQ1 | Provinsi baku, jumlah provinsi, kelompok pulau, pewarisan lokasi dari konteks | `rq2_*` |
| **RQ3:** kekhawatiran/keluhan publik | Sentiment analysis | Teks minimal (negasi dan emoji dipertahankan), tingkat kelayakan, sampel pelabelan manual | `rq3_*` |

### Input dan output

| | Berkas |
|---|---|
| Input | `data/processed/raw_merged/mbg_merged.csv` (+ `merge_metadata.json`), hasil penggabungan data X dan YouTube |
| Output | `data/processed/mbg_processed/mbg_processed.csv`, `data_dictionary.csv`, `preprocessing_log.csv` |

### Keputusan desain

1. **Satu dataset untuk ketiga RQ.** Tidak ada dataset terpisah per RQ. Hasil olahan berupa **kolom tambahan** pada dataset yang sama, dan baris tidak dibuang. Alasannya: langkah umum (duplikat, spam, relevansi, lokasi) dipakai bersama, RQ2 membutuhkan hasil RQ1, dan satu `doc_id` cukup untuk melacak seluruh perlakuan.
2. **Pembeda antar-RQ ada di kolom, bukan di baris.** Setiap RQ punya penanda kelayakan (`rq1_eligible`, `rq2_eligible`, `rq3_eligible`); potongan data dibuat saat analisis (misalnya `df[df.rq1_eligible]`).
3. **Kolom asli tidak diubah.** Seluruh 24 kolom dari merge tetap utuh (diverifikasi dengan hash). Kolom baru memakai awalan `common_`, `rq1_`, `rq2_`, `rq3_`.
4. **Teks berbeda untuk tiap RQ.** Topic modelling membuang stopword dan melakukan stemming, sedangkan sentimen justru membutuhkan negasi ("tidak", "bukan"), emoji, dan tanda seru. Karena itu ada tiga bentuk teks (`rq1_text_light`, `rq1_tokens`, `rq3_text`); kolom teks hanya diisi untuk dokumen yang layak agar berkas tidak membengkak.
5. **Keputusan yang diambil:** (a) RQ1 menyiapkan dua bentuk teks dan memilih model belakangan; (b) RQ3 disiapkan untuk model IndoBERT dan dilengkapi sampel pelabelan manual; (c) RQ3 memakai dua tingkat kelayakan: **A** (utama, menyebut keamanan pangan sendiri) dan **B** (sensitivitas, konteksnya yang membahas keamanan pangan); (d) lokasi YouTube boleh diwarisi dari video induk dengan syarat ketat dan disimpan di kolom terpisah; (e) berkas data besar tidak di-commit ke Git (`.gitignore`).

### Cara menjalankan

1. Pastikan `mbg_merged.csv` ada di `data/processed/raw_merged/`.
2. Jalankan seluruh sel dari atas (**Run All**). Sel 1.1 memerlukan `pandas`, `emoji`, dan `Sastrawi` (`pip install pandas emoji Sastrawi`). Seluruh proses memakan waktu sekitar 1-2 menit.
3. Untuk uji cepat, ubah `NROWS` di sel 1.1 menjadi angka (misalnya `50000`); nilai `None` memakai seluruh data.
4. Notebook dapat dijalankan ulang jika data di-merge ulang (mis. koleksi YouTube bertambah), karena angka dihitung dari berkas, bukan ditulis manual.

> **Catatan.** Berkas data besar (`mbg_merged.csv` sekitar 370 MB dan `mbg_processed.csv` sekitar 550 MB) sudah dimasukkan ke `.gitignore`. Teks komentar dan unggahan bersifat publik tetapi masih dapat memuat `@mention` di `text_raw`/`text_clean`; kolom `rq1_text_light` tidak memuat mention, dan `rq3_text` mengganti seluruh mention dengan `@user`. Tampilan contoh di notebook ini menyamarkan mention.

# 1. Persiapan dan Audit Input

## 1.1. Import Library dan Konfigurasi

In [1]:
import hashlib
import html
import json
import re
import time
import unicodedata
from collections import Counter
from pathlib import Path

import emoji
import numpy as np
import pandas as pd
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

pd.set_option("display.max_colwidth", 110)
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)

SEED = 42
NROWS = None  # isi angka (mis. 50000) untuk uji cepat pada sampel acak; None = seluruh data

BASE = Path.cwd().parent if Path.cwd().name == "rq_preprocessing" else Path.cwd()
IN_DIR = BASE / "data" / "processed" / "raw_merged"
IN_CSV = IN_DIR / "mbg_merged.csv"
IN_META = IN_DIR / "merge_metadata.json"
OUT_DIR = BASE / "data" / "processed" / "mbg_processed"
OUT_CSV = OUT_DIR / "mbg_processed.csv"
OUT_DICT = OUT_DIR / "data_dictionary.csv"
OUT_LOG = OUT_DIR / "preprocessing_log.csv"
OUT_DIR.mkdir(parents=True, exist_ok=True)

MIN_WORDS = 3                      # minimal kata agar teks dianggap cukup panjang (umum)
MIN_RQ1_TOKENS = 3                 # minimal token hasil stemming agar layak untuk topic modelling
LABEL_SAMPLE_PER_PLATFORM = 400    # ukuran sampel pelabelan manual sentimen (RQ3)
AUDIT_SAMPLE_PER_GROUP = 100       # sampel audit flag relevansi, per platform dan per nilai flag

log = []


def aman(seri, n=90):
    """Hanya untuk tampilan contoh: mention (@akun) disamarkan menjadi @user dan teks dipotong."""
    return seri.str.replace(r"@\w+", "@user", regex=True).str.slice(0, n)


def catat(langkah, keterangan, n_baris=None, n_ditandai=None):
    """Satu baris log pre-processing: n_baris = baris yang diproses, n_ditandai = baris yang bernilai True/terpengaruh."""
    log.append({"langkah": langkah, "keterangan": keterangan, "n_baris": n_baris, "n_ditandai": n_ditandai})

Seluruh parameter ada di sel ini agar mudah dicatat di laporan dan diubah: ambang jumlah kata, ukuran sampel audit dan pelabelan, serta `NROWS` untuk uji cepat. Fungsi `catat()` merekam setiap langkah ke log pre-processing, dan `aman()` menyamarkan `@mention` pada tampilan contoh.

## 1.2. Muat Dataset Gabungan dan Audit terhadap Metadata

In [2]:
meta = json.loads(IN_META.read_text(encoding="utf-8"))
t0 = time.time()
df = pd.read_csv(IN_CSV, dtype=str, keep_default_na=False, na_values=[""])
if NROWS:
    df = df.sample(NROWS, random_state=SEED).sort_index().reset_index(drop=True)
print(f"Dimensi dataset gabungan (baris, kolom): {df.shape} | dibaca dalam {time.time() - t0:.0f} detik")

BOOL_COLS = ["is_suspected_spam", "is_repeated_text", "mentions_mbg", "mentions_food_safety", "context_mentions_mbg", "is_relevant_candidate"]
for kolom in BOOL_COLS:
    assert set(df[kolom].dropna().unique()) <= {"True", "False"}, kolom
    df[kolom] = df[kolom].map({"True": True, "False": False}).astype(bool)
df["like_count"] = pd.to_numeric(df["like_count"]).astype("int64")
df["reply_count"] = pd.to_numeric(df["reply_count"]).astype("Int64")  # kosong pada reply dan video YouTube
ORIG_COLS = list(df.columns)
HASH_ASLI = int(pd.util.hash_pandas_object(df[ORIG_COLS], index=False).sum() % (2**63))  # untuk memastikan kolom asli tidak berubah

print("\nKolom asli:", len(ORIG_COLS))
print(pd.crosstab(df["platform"], df["doc_type"]))

if not NROWS:
    cek_meta = {
        "jumlah baris total sesuai merge_metadata.json": len(df) == meta["rows"]["total"],
        "jumlah baris X dan YouTube sesuai": df["platform"].value_counts().to_dict() == {"youtube": meta["rows"]["youtube"], "x": meta["rows"]["x"]},
        "doc_id unik": bool(df["doc_id"].is_unique),
    }
    for flag in BOOL_COLS:
        if flag in meta["flags"]:
            hitung = df.groupby("platform")[flag].sum().to_dict()
            cek_meta[f"hitungan {flag} sesuai metadata"] = hitung == meta["flags"][flag]
    for nama, ok in cek_meta.items():
        print("PASS" if ok else "FAIL", "-", nama)
catat("1.2 Muat dataset gabungan", "mbg_merged.csv; tipe bool dan angka dikonversi, kolom asli tidak diubah isinya", len(df))

Dimensi dataset gabungan (baris, kolom): (1109888, 24) | dibaca dalam 12 detik

Kolom asli: 24
doc_type  comment   post  quote   reply  video
platform                                      
x               0  13985   2481    8069      0
youtube    892141      0      0  175549  17663
PASS - jumlah baris total sesuai merge_metadata.json
PASS - jumlah baris X dan YouTube sesuai
PASS - doc_id unik
PASS - hitungan is_suspected_spam sesuai metadata
PASS - hitungan is_repeated_text sesuai metadata
PASS - hitungan mentions_mbg sesuai metadata
PASS - hitungan mentions_food_safety sesuai metadata
PASS - hitungan context_mentions_mbg sesuai metadata
PASS - hitungan is_relevant_candidate sesuai metadata


Dataset gabungan berisi **669.383 dokumen** dan 24 kolom: 24.535 dari X dan 644.848 dari YouTube (96%). Seluruh pemeriksaan terhadap `merge_metadata.json` **lulus**: jumlah baris, komposisi platform, `doc_id` unik, dan hitungan keenam flag dari merge identik dengan yang dicatat di metadata. Tipe data flag diubah menjadi `bool` dan `like_count`/`reply_count` menjadi angka, sedangkan isi kolom asli tidak diubah (hash isinya dicatat untuk diverifikasi di bagian 6.2).

## 1.3. Pemeriksaan Kondisi Data Masukan

In [3]:
pub = pd.to_datetime(df["published_at_utc"], utc=True)
print("Rentang waktu terbit per platform (UTC):")
print(pub.groupby(df["platform"]).agg(["min", "max"]).to_string())
print("\nMissing value per kolom (hanya yang ada):")
miss = df.isna().sum()
print(miss[miss > 0].to_string())
print("\nMissing menurut platform dan jenis dokumen:")
for kolom in ["parent_doc_id", "author_hash", "reply_count", "collected_at_utc"]:
    print(f"  {kolom:18s}", df[df[kolom].isna()].groupby(["platform", "doc_type"]).size().to_dict())
loc_tanpa_prov = df["location_names"].notna() & df["province"].isna()
print("\nDokumen dengan location_names tetapi tanpa province:", int(loc_tanpa_prov.sum()), "| nama lokasi:", df.loc[loc_tanpa_prov, "location_names"].value_counts().to_dict())
print("Reply YouTube: induk ada di dataset =", round(df[(df.doc_type == "reply") & (df.platform == "youtube")]["parent_doc_id"].isin(df["doc_id"]).mean(), 3),
      "| reply X: induk ada di dataset =", round(df[(df.doc_type == "reply") & (df.platform == "x")]["parent_doc_id"].isin(df["doc_id"]).mean(), 3))
print("Komentar/reply YouTube: context_id (video) ada di dataset =", round(df[(df.platform == "youtube") & (df.doc_type != "video")]["context_id"].isin(df.loc[df.doc_type == "video", "doc_id"]).mean(), 3))
print("Panjang text_clean (karakter), median per platform dan jenis:")
print(df["text_clean"].str.len().groupby([df["platform"], df["doc_type"]]).median().to_string())

Rentang waktu terbit per platform (UTC):
                               min                       max
platform                                                    
x        2025-01-01 07:56:38+00:00 2026-09-29 23:02:39+00:00
youtube  2025-01-05 22:34:42+00:00 2026-10-24 00:55:09+00:00

Missing value per kolom (hanya yang ada):
parent_doc_id        31839
source_name          24535
source_category      24535
author_hash          17663
reply_count         175555
location_names     1075231
province           1075231
quoted_doc_id      1107134

Missing menurut platform dan jenis dokumen:
  parent_doc_id      {('x', 'post'): 13985, ('x', 'quote'): 17, ('x', 'reply'): 174, ('youtube', 'video'): 17663}
  author_hash        {('youtube', 'video'): 17663}
  reply_count        {('youtube', 'reply'): 175549, ('youtube', 'video'): 6}
  collected_at_utc   {}

Dokumen dengan location_names tetapi tanpa province: 0 | nama lokasi: {}
Reply YouTube: induk ada di dataset = 1.0 | reply X: induk ada di datas

Beberapa hal yang perlu diperhatikan pada data masukan:

* **Cakupan waktu berbeda.** X mencakup 1 Januari 2025–29 September 2026, YouTube 5 Januari 2025–22 Oktober 2026. Sesuai `merge_metadata.json`, koleksi YouTube juga belum final (`youtube_collection_complete: false`).
* **Missing value bersifat struktural.** `author_hash` kosong pada seluruh video (11.063), `reply_count` kosong pada 107.505 reply YouTube dan 6 video, dan `parent_doc_id` kosong pada post X, video, serta 174 reply dan 17 quote X yang induknya tidak terbaca. Satu komentar YouTube tidak memiliki `collected_at_utc`.
* **Keterhubungan konteks.** Seluruh reply YouTube memiliki induk di dataset dan seluruh komentar/reply memiliki video-nya, sehingga konteks video dapat dipakai penuh. Pada X hanya 13,6% reply yang induknya ada di dataset.
* **16 dokumen** memiliki `location_names` = "Banjar" tetapi tanpa `province`, karena nama ini ambigu (Kota Banjar di Jawa Barat atau Kabupaten Banjar di Kalimantan Selatan). Dokumen ini tidak diberi provinsi di bagian 4.
* **Panjang teks sangat berbeda:** komentar YouTube bermedian 66 karakter, unggahan X 209–269 karakter, dan deskripsi video 1.065 karakter.

# 2. Pre-Processing Umum (kolom `common_`)

Bagian ini membuat penanda dan fitur yang dipakai **bersama oleh ketiga RQ** (kolom `common_`). Tidak ada baris yang dibuang: dokumen bermasalah hanya ditandai, lalu penanda digabung menjadi `common_is_usable`. Perbandingan teks memakai bentuk **NFKC + huruf kecil** yang hanya dibuat sementara di memori (tidak disimpan), karena banyak komentar memakai huruf Unicode "fancy" (misalnya 𝗠𝗕𝗚) yang tidak terbaca oleh pencocokan kata biasa.

## 2.1. Waktu Terbit

In [4]:
df["common_published_date_wib"] = pub.dt.tz_convert("Asia/Jakarta").dt.strftime("%Y-%m-%d")
df["common_year_month"] = pub.dt.tz_convert("Asia/Jakarta").dt.strftime("%Y-%m")

batas = pub.groupby(df["platform"]).agg(["min", "max"])
tumpang_mulai, tumpang_akhir = batas["min"].max(), batas["max"].min()
df["common_in_overlap_period"] = pub.between(tumpang_mulai, tumpang_akhir)
print(f"Periode tumpang tindih X dan YouTube: {tumpang_mulai} -> {tumpang_akhir}")
print("Dokumen di dalam periode tumpang tindih:", df.groupby("platform")["common_in_overlap_period"].agg(["sum", "mean"]).round(3).to_dict("index"))
print("\nDokumen per bulan (WIB) dan platform:")
print(pd.crosstab(df["common_year_month"], df["platform"]).T.to_string())
catat("2.1 Waktu", "common_published_date_wib, common_year_month, common_in_overlap_period", len(df), int(df["common_in_overlap_period"].sum()))

Periode tumpang tindih X dan YouTube: 2025-01-05 22:34:42+00:00 -> 2026-09-29 23:02:39+00:00
Dokumen di dalam periode tumpang tindih: {'x': {'sum': 24513, 'mean': 0.999}, 'youtube': {'sum': 1082389, 'mean': 0.997}}

Dokumen per bulan (WIB) dan platform:
common_year_month  2025-01  2025-02  2025-03  2025-04  2025-05  2025-06  2025-07  2025-08  2025-09  2025-10  2025-11  2025-12  2026-01  2026-02  2026-03  2026-04  2026-05  2026-06  2026-07  2026-08  2026-09  2026-10
platform                                                                                                                                                                                                               
x                     1059      571      331      546      519      480      422     1031     1284     2503      928      763     1800     1789     1719     1758     1583     1670      677      867     2235        0
youtube              60635    60805    15794    31903    12243     1610     2617     8883    87264

Kolom `common_published_date_wib` dan `common_year_month` memakai zona waktu WIB. Sebanyak 99,9% dokumen berada di dalam periode tumpang tindih X dan YouTube (5 Januari 2025–29 September 2026), ditandai `common_in_overlap_period` untuk perbandingan antarplatform.

Jumlah dokumen per bulan **sangat tidak merata**, terutama pada YouTube: dari 1.376 dokumen (Juni 2025) hingga **144.729 dokumen (Juni 2026)**, sedangkan X berkisar 331–2.503 per bulan. Ketimpangan ini berasal dari cara koleksi (dan, pada X, dari cakupan pengumpulan yang dibahas di bagian 2.2.5 notebook pengumpulan), bukan dari perilaku pengguna, sehingga **perbandingan volume antar bulan tidak boleh dibaca sebagai tren percakapan** dan analisis waktu sebaiknya memakai proporsi atau diberi bobot.

## 2.2. Normalisasi Dasar dan Panjang Teks

In [5]:
RE_URL = re.compile(r"https?://\S+|www\.\S+", re.I)
RE_MENTION = re.compile(r"@\w+")
RE_HASHTAG = re.compile(r"#(\w+)")
RE_INVISIBLE = re.compile("[\u00ad\u200b-\u200f\u202a-\u202e\u2060-\u2064\ufeff]")  # termasuk ZWJ (U+200D); dipakai hanya untuk bahan hitung
RE_NONWORD = re.compile(r"[\W_]+")


def norm_dasar(teks):
    """Bentuk pembanding saja (tidak disimpan): NFKC, tanpa karakter tak terlihat, huruf kecil."""
    return RE_INVISIBLE.sub("", unicodedata.normalize("NFKC", teks)).lower()


df["_norm"] = df["text_clean"].map(norm_dasar)
df["_plain"] = df["_norm"].map(lambda s: RE_NONWORD.sub(" ", RE_MENTION.sub(" ", RE_URL.sub(" ", s))).strip())

ubah_nfkc = (df["text_clean"].str.lower() != df["_norm"])
print("Teks yang berubah oleh NFKC (misalnya huruf 'fancy' seperti 𝗠𝗕𝗚):", int(ubah_nfkc.sum()), f"({ubah_nfkc.mean():.1%})")
contoh = df[ubah_nfkc & (df["doc_type"] != "video")].sample(4, random_state=SEED)
pd.DataFrame({"text_clean": aman(contoh["text_clean"], 70).values, "setelah NFKC": aman(contoh["_norm"], 70).values})

Teks yang berubah oleh NFKC (misalnya huruf 'fancy' seperti 𝗠𝗕𝗚): 59055 (5.3%)


,text_clean,setelah NFKC
0,​@user lo teriak stop mbg ke anak2 daerah ....ke petani...nelayan....p,@user lo teriak stop mbg ke anak2 daerah ....ke petani...nelayan....pe
1,"​Ya elah yat rakyat, kenapa sih Bahlil serba salah, pahami dong","ya elah yat rakyat, kenapa sih bahlil serba salah, pahami dong"
2,​@user datang ke kampung2x sana. Jgn ngomong data. Lihat anak2x itu be,@user datang ke kampung2x sana. jgn ngomong data. lihat anak2x itu bet
3,​@user setuju dengan pertanyaan ini 👍🏻,@user setuju dengan pertanyaan ini 👍🏻


Normalisasi NFKC mengubah **35.801 teks (5,3%)**, sebagian besar komentar yang memakai huruf matematika Unicode, huruf lebar penuh, atau gaya huruf lain. Tanpa langkah ini, kata seperti "𝗠𝗕𝗚" atau "𝗞𝗲𝗿𝗮𝗰𝘂𝗻𝗮𝗻" tidak terdeteksi oleh pencocokan kata kunci, deteksi duplikat, maupun stemming.

In [6]:
df["common_text_len"] = df["text_clean"].str.len()
df["common_n_words"] = df["_plain"].str.count(r"[^\W\d_]+").astype("int32")
df["common_is_too_short"] = df["common_n_words"] < MIN_WORDS
print(f"Dokumen dengan kurang dari {MIN_WORDS} kata (common_is_too_short):", int(df["common_is_too_short"].sum()))
print(df.groupby(["platform", "doc_type"])["common_is_too_short"].mean().round(3).to_string())
print("\nJumlah kata per dokumen (median, persentil ke-95):")
print(df.groupby(["platform", "doc_type"])["common_n_words"].describe(percentiles=[0.5, 0.95])[["50%", "95%", "max"]].to_string())
catat("2.2 Normalisasi dasar dan panjang teks", "common_text_len, common_n_words, common_is_too_short", len(df), int(df["common_is_too_short"].sum()))

Dokumen dengan kurang dari 3 kata (common_is_too_short): 90028
platform  doc_type
x         post        0.001
          quote       0.002
          reply       0.003
youtube   comment     0.078
          reply       0.118
          video       0.000

Jumlah kata per dokumen (median, persentil ke-95):
                     50%    95%     max
platform doc_type                      
x        post       30.0  209.0  1443.0
         quote      34.0  197.0  1025.0
         reply      31.0   55.0   520.0
youtube  comment    10.0   44.0  1475.0
         reply      10.0   50.0  1611.0
         video     121.0  284.0   684.0


Sebanyak **53.934 dokumen (8,06%)** memiliki kurang dari 3 kata (`common_is_too_short`). Hampir seluruhnya komentar dan reply YouTube (7,8% dan 12,0%), sedangkan pada X kurang dari 0,5%. Dokumen sependek ini (misalnya "hmm" atau emoji saja) tidak cukup informatif untuk topic modelling maupun klasifikasi sentimen. Median jumlah kata juga menunjukkan perbedaan karakter data: 10 kata pada komentar YouTube, sekitar 30 pada X, dan 115 pada deskripsi video.

## 2.3. Spam

In [7]:
# Tanda spam dari merge dipertahankan; diperluas ke dokumen lain yang memuat "merek" situs judi yang sama (huruf+angka), setelah NFKC
RE_MEREK = re.compile(r"\b[a-z]{3,}\d{2,4}\b")
spam_merge = df["is_suspected_spam"]
hitung_merek = Counter(m for teks in df.loc[spam_merge, "_plain"] for m in set(RE_MEREK.findall(teks)))
merek = {m for m, c in hitung_merek.items() if c >= 5}
print("Merek situs judi yang berulang pada dokumen spam:", sorted(merek)[:15], "..." if len(merek) > 15 else "")
RE_MEREK_SPAM = re.compile(r"\b(?:" + "|".join(map(re.escape, sorted(merek))) + r")\b") if merek else None
mengandung_merek = df["_plain"].str.contains(RE_MEREK_SPAM) if RE_MEREK_SPAM else pd.Series(False, index=df.index)
df["common_is_spam"] = spam_merge | mengandung_merek
print(f"is_suspected_spam (dari merge)      : {int(spam_merge.sum())}")
print(f"Tambahan karena memuat merek yang sama: {int((mengandung_merek & ~spam_merge).sum())}")
print(f"common_is_spam (gabungan)            : {int(df['common_is_spam'].sum())} ({df['common_is_spam'].mean():.2%})")
print(df[df["common_is_spam"]].groupby(["platform", "doc_type"]).size().to_dict())
catat("2.3 Spam", "common_is_spam = is_suspected_spam OR memuat merek situs judi yang berulang pada dokumen spam", len(df), int(df["common_is_spam"].sum()))

Merek situs judi yang berulang pada dokumen spam: ['aero88', 'berkah99', 'cuan328', 'dora77', 'gel77', 'istanabet17', 'jptogel77', 'macau328', 'mandalika77', 'miya88', 'pstoto99', 'suhu328', 'toto328', 'win303'] 
is_suspected_spam (dari merge)      : 2357
Tambahan karena memuat merek yang sama: 9
common_is_spam (gabungan)            : 2366 (0.21%)
{('youtube', 'comment'): 2363, ('youtube', 'reply'): 3}


Flag spam dari merge (2.089 dokumen) dipertahankan dan diperluas dengan dokumen yang memuat "merek" situs judi yang sama (pola huruf+angka seperti `win303` atau `jptogel77`, setelah NFKC). Penambahan hanya **3 dokumen**, sehingga total `common_is_spam` adalah **2.092 (0,31%)**, seluruhnya di YouTube. Flag dari merge sudah mencakup hampir semua kasus; polanya data-driven dan tidak mencoba menebak merek yang belum pernah muncul pada dokumen spam.

## 2.4. Teks Berulang

In [8]:
kunci = df["_plain"].where(df["_plain"] != "", df["_norm"])  # teks tanpa huruf (misalnya hanya emoji) memakai bentuk NFKC-nya
tmp = pd.DataFrame({"kunci": kunci, "penulis": df["author_hash"], "waktu": pub, "doc_id": df["doc_id"]}, index=df.index)
tmp = tmp.sort_values(["waktu", "doc_id"], kind="stable")
grup = tmp.groupby("kunci", sort=False)
tmp["urut"] = grup.cumcount() + 1
tmp["ukuran"] = grup["kunci"].transform("size")
tmp["penulis_ulang"] = tmp["penulis"].notna() & tmp.duplicated(["kunci", "penulis"], keep="first")  # penulis sama mengulang teks yang sama
tmp = tmp.reindex(df.index)

df["common_dup_group_size"] = tmp["ukuran"].astype("int32")
df["common_is_dup_first"] = tmp["urut"] == 1
df["common_is_dup_same_author"] = tmp["penulis_ulang"]
del tmp

berulang = df["common_dup_group_size"] > 1
print(f"Dokumen dalam grup teks yang sama (ukuran > 1): {int(berulang.sum())} ({berulang.mean():.1%}) dalam {int((berulang & df['common_is_dup_first']).sum())} grup")
print("Pengulangan oleh penulis yang sama (common_is_dup_same_author):", int(df["common_is_dup_same_author"].sum()))
print("Perbandingan dengan flag is_repeated_text dari merge:")
print(pd.crosstab(df["is_repeated_text"], berulang, rownames=["is_repeated_text (merge)"], colnames=["grup teks > 1 (kolom ini)"]).to_string())
print("\nTeks terbanyak diulang:")
top = df[df["common_is_dup_first"] & (df["common_dup_group_size"] > 50)].sort_values("common_dup_group_size", ascending=False)
pd.DataFrame({"teks": aman(top["text_clean"], 60).values, "jumlah": top["common_dup_group_size"].values}).head(8)

Dokumen dalam grup teks yang sama (ukuran > 1): 95629 (8.6%) dalam 19666 grup
Pengulangan oleh penulis yang sama (common_is_dup_same_author): 19548
Perbandingan dengan flag is_repeated_text dari merge:
grup teks > 1 (kolom ini)    False  True 
is_repeated_text (merge)                 
False                      1014259  24880
True                             0  70749

Teks terbanyak diulang:


,teks,jumlah
0,STOP MBG.....!!!,1813
1,Setuju,1196
2,Mantap,1002
3,😂😂😂,851
4,😂😂😂🤣🤣🤣betul,762
5,😂,718
6,Omon omon..,668
7,Bubarkan mbg,666


Sebanyak **53.043 dokumen (7,9%)** berada dalam grup teks yang sama, dengan **11.546 grup**. Seluruh 39.295 dokumen berflag `is_repeated_text` dari merge termasuk di sini; tambahan 13.748 dokumen muncul karena kunci pembanding di sini lebih longgar (setelah NFKC, tanpa URL, mention, dan tanda baca), sehingga teks yang hanya berbeda spasi atau tanda baca ikut terdeteksi.

Tiga kolom dibuat: `common_dup_group_size` (bisa dipakai sebagai bobot), `common_is_dup_first` (kemunculan pertama menurut waktu), dan `common_is_dup_same_author` (**10.261 dokumen** yang merupakan pengulangan teks oleh penulis yang sama). Perlakuan berbeda antar-RQ: pengulangan penulis yang sama dikeluarkan dari semua RQ (ciri spam atau unggahan ganda), sedangkan teks yang sama dari **penulis berbeda** hanya dibuang dari RQ1 (agar topik tidak menggembung), tetapi dipertahankan pada RQ3 karena merupakan opini yang sama dari orang yang berbeda.

In [9]:
catat("2.4 Teks berulang", "common_dup_group_size, common_is_dup_first, common_is_dup_same_author; tidak ada baris dibuang", len(df), int(df["common_is_dup_same_author"].sum()))

## 2.5. Bahasa

In [10]:
# Deteksi bahasa berbasis aturan sederhana. Pendeteksi umum (misalnya langid) tidak dipakai karena sering menebak bahasa gaul Indonesia sebagai Melayu atau Inggris.
KATA_INGGRIS = set("""the is are was were and of to you that this it we they for with not have has be but what why how who my your our
will would can could just so if from all more very do does did""".split())


def bagian_inggris(teks):
    kata = teks.split()
    return sum(k in KATA_INGGRIS for k in kata) / len(kata) if len(kata) >= 3 else 0.0


huruf = df["_plain"].str.count(r"[^\W\d_]")
huruf_latin = df["_plain"].str.count(r"[A-Za-z\u00C0-\u024F]")
bukan_latin = (huruf >= 5) & ((huruf - huruf_latin) / huruf.where(huruf > 0, 1) > 0.5)  # Arab, Mandarin, Cyrillic, dan sejenisnya
inggris = df["_plain"].map(bagian_inggris) >= 0.30
df["common_is_non_indonesian"] = bukan_latin | inggris
print(f"Berbahasa Inggris (>= 30% kata fungsi Inggris, minimal 3 kata): {int(inggris.sum())} | aksara non-Latin dominan: {int(bukan_latin.sum())}")
print(f"common_is_non_indonesian: {int(df['common_is_non_indonesian'].sum())} ({df['common_is_non_indonesian'].mean():.2%})")
print(df.groupby("platform")["common_is_non_indonesian"].agg(["sum", "mean"]).round(4).to_string())
print("\nContoh yang ditandai (untuk pemeriksaan manual):")
tampil = aman(df[df["common_is_non_indonesian"]]["text_clean"], 90)
print(tampil.sample(min(8, len(tampil)), random_state=SEED).tolist())
catat("2.5 Bahasa", "common_is_non_indonesian: dominan kata fungsi Inggris atau aksara non-Latin", len(df), int(df["common_is_non_indonesian"].sum()))

Berbahasa Inggris (>= 30% kata fungsi Inggris, minimal 3 kata): 1008 | aksara non-Latin dominan: 127
common_is_non_indonesian: 1135 (0.10%)
           sum   mean
platform             
x            0  0.000
youtube   1135  0.001

Contoh yang ditandai (untuk pemeriksaan manual):
['METRO IS COOKING 🔥🔥🔥', 'Knapa ngomong nya so fast??? You talk to fast!', 'This the most worst president of indonesia raya, and same worst with joko', 'How dare you🥹', 'keep it up', 'mana bisa the and...... Tapi To Be Continue wong programnya masih jalan', '🏋\u200d♀🤔☠️🕸🛶💣welcome to the Jungle 🎉', 'Tho To The Lol. An To The Njeeenk. Bang To The Saaat.']


Deteksi bahasa memakai aturan sederhana: dokumen dianggap bukan bahasa Indonesia jika didominasi kata fungsi Inggris (minimal 30%) atau beraksara non-Latin. Hasilnya **700 dokumen (0,10%)**, seluruhnya di YouTube (620 berbahasa Inggris dan 80 beraksara non-Latin). Pendeteksi bahasa umum (misalnya `langid`) sempat dicoba tetapi **tidak dipakai**, karena menebak bahasa gaul Indonesia sebagai Melayu atau Inggris sehingga akan membuang banyak dokumen Indonesia yang valid. Bahasa daerah (Jawa, Sunda) tidak dikeluarkan; contoh yang tampil menunjukkan kadang tercampur satu-dua teks berbahasa daerah, yang jumlahnya sangat kecil.

## 2.6. Konteks Keamanan Pangan

In [11]:
# Konteks = video (YouTube) atau unggahan akar yang ada di dataset (X); dokumen yang konteksnya adalah dirinya sendiri tidak punya konteks
induk_fs = pd.Series(df["mentions_food_safety"].to_numpy(), index=df["doc_id"])
konteks_fs = df["context_id"].map(induk_fs)
punya_konteks = (df["context_id"] != df["doc_id"]) & konteks_fs.notna()
df["common_context_mentions_food_safety"] = (konteks_fs.where(punya_konteks, False)).astype(bool)

print("context_mentions_mbg (dari merge) tidak membedakan: True pada", f"{df.groupby('platform')['context_mentions_mbg'].mean().round(3).to_dict()}")
print("common_context_mentions_food_safety:")
print(df[df["doc_type"] != "video"].groupby("platform")["common_context_mentions_food_safety"].agg(["sum", "mean"]).round(3).to_string())
print("\nKomentar tanpa kata kunci pangan sendiri, tetapi video/konteksnya membahas keamanan pangan:", int((df["common_context_mentions_food_safety"] & ~df["mentions_food_safety"] & (df["doc_type"] != "video")).sum()))
catat("2.6 Konteks keamanan pangan", "common_context_mentions_food_safety (video atau unggahan akar menyebut kata kunci keamanan pangan)", len(df), int(df["common_context_mentions_food_safety"].sum()))

context_mentions_mbg (dari merge) tidak membedakan: True pada {'x': 0.711, 'youtube': 0.998}
common_context_mentions_food_safety:
             sum   mean
platform               
x            656  0.027
youtube   261731  0.245

Komentar tanpa kata kunci pangan sendiri, tetapi video/konteksnya membahas keamanan pangan: 203834


`context_mentions_mbg` dari merge bernilai True pada 99,6% dokumen YouTube sehingga tidak membedakan apa pun. Karena itu dibuat `common_context_mentions_food_safety` yang lebih selektif: konteks (video atau unggahan akar) menyebut kata kunci keamanan pangan. Hasilnya True pada **26,4% dokumen YouTube (167.175)** tetapi hanya 2,7% pada X, karena pada X konteksnya jarang ada di dataset. Sebanyak **129.946 komentar** tidak menyebut kata kunci pangan sendiri tetapi berada di bawah video yang membahasnya; ini adalah kandidat tingkat B pada RQ3.

## 2.7. Dokumen Layak Pakai (`common_is_usable`)

In [12]:
df["common_is_usable"] = ~df["common_is_spam"] & ~df["common_is_too_short"] & ~df["common_is_non_indonesian"] & ~df["common_is_dup_same_author"]
alasan = pd.DataFrame({
    "spam": df["common_is_spam"], "terlalu pendek": df["common_is_too_short"],
    "bukan bahasa Indonesia": df["common_is_non_indonesian"], "pengulangan penulis sama": df["common_is_dup_same_author"],
})
tabel = pd.DataFrame({"jumlah": alasan.sum(), "persen": (alasan.mean() * 100).round(2)})
print(tabel.to_string())
print(f"\ncommon_is_usable: {int(df['common_is_usable'].sum())} dari {len(df)} ({df['common_is_usable'].mean():.1%})")
print(df.groupby("platform")["common_is_usable"].mean().round(3).to_dict())
print("Di antara dokumen relevan (is_relevant_candidate):", f"{df.loc[df['is_relevant_candidate'], 'common_is_usable'].mean():.1%} layak")
catat("2.7 Dokumen layak pakai", "common_is_usable = bukan spam, cukup panjang, bahasa Indonesia, bukan pengulangan penulis yang sama", len(df), int(df["common_is_usable"].sum()))

                          jumlah  persen
spam                        2366    0.21
terlalu pendek             90028    8.11
bukan bahasa Indonesia      1135    0.10
pengulangan penulis sama   19548    1.76

common_is_usable: 1001029 dari 1109888 (90.2%)
{'x': 0.981, 'youtube': 0.9}
Di antara dokumen relevan (is_relevant_candidate): 96.8% layak


`common_is_usable` bernilai True pada **604.616 dokumen (90,3%)**: 98,1% pada X dan 90,0% pada YouTube. Alasan terbanyak adalah teks terlalu pendek (8,06%), diikuti pengulangan oleh penulis yang sama (1,53%), spam (0,31%), dan bukan bahasa Indonesia (0,10%); alasan ini dapat bertumpuk pada satu dokumen. Di antara dokumen yang berflag relevan, **97,3% layak pakai**, sehingga penyaringan umum ini hampir tidak mengurangi data inti.

## 2.8. Sampel Audit Relevansi

In [13]:
# Sampel audit untuk memeriksa ketepatan flag relevansi secara manual: per platform dan per nilai is_relevant_candidate
calon = df[(df["doc_type"] != "video") & ~df["common_is_spam"]]
audit_idx = (
    calon.groupby(["platform", "is_relevant_candidate"], group_keys=False)
    .sample(AUDIT_SAMPLE_PER_GROUP, random_state=SEED)
    .index
)
df["common_audit_sample"] = df.index.isin(audit_idx)
print("Sampel audit relevansi:", int(df["common_audit_sample"].sum()), "dokumen")
print(df[df["common_audit_sample"]].groupby(["platform", "is_relevant_candidate"]).size().to_string())
print("\nCuplikan dokumen berflag relevan (untuk pemeriksaan cepat; audit penuh dilakukan manual):")
for p in ["x", "youtube"]:
    for teks in aman(df[df["common_audit_sample"] & (df["platform"] == p) & df["is_relevant_candidate"]]["text_clean"].head(4), 120):
        print(f"  [{p}] {teks}")
catat("2.8 Sampel audit relevansi", "common_audit_sample (100 dokumen per platform per nilai is_relevant_candidate)", len(df), int(df["common_audit_sample"].sum()))

Sampel audit relevansi: 400 dokumen
platform  is_relevant_candidate
x         False                    100
          True                     100
youtube   False                    100
          True                     100

Cuplikan dokumen berflag relevan (untuk pemeriksaan cepat; audit penuh dilakukan manual):
  [x] Komentar Istana soal Kasus Siswa SDN Sukoharjo Keracunan Makan Bergizi Gratis https://www.kompas.tv/nasional/567812/kome
  [x] Program makan bergizi gratis sajikan makanan berkualitas dan higienis #makanbegizigratis #visipresidenprabowo #indonesia
  [x] Kepala BGN: Serangga dan Ulat Bisa Jadi 'Sumber Protein' Untuk Makan Bergizi Gratis https://www.democrazy.id/2025/01/kep
  [x] Kejadian lg MBG berulat. Setelah MBG jadi makan beracun gratis skrng tambah lg, makan berulat gratis. Mba @user sdh inga
  [youtube] DPR wee. ...Bu..biar semakin muntah...kelolokeun uang haramm
  [youtube] Kepala badan gizi tau soal gizi ga,jangan samain orang sama serangga
  [youtube] Udah lah di

Sampel audit berisi 400 dokumen (100 per platform per nilai `is_relevant_candidate`) untuk memeriksa **ketepatan flag relevansi secara manual**. Pemeriksaan sekilas pada cuplikan acak menunjukkan sebagian besar dokumen berflag relevan memang membicarakan keamanan pangan MBG, tetapi ada yang meleset karena kata umum dipakai figuratif (misalnya "bau koruptor" atau "piring kotor"). **Presisi dan recall flag belum diukur secara formal**; pelabelan manual sampel ini perlu dilakukan sebelum hasil analisis diklaim.

# 3. Pre-Processing RQ1: Topic Modelling (kolom `rq1_`)

Untuk RQ1 disiapkan **dua bentuk teks**, karena model topik belum dipilih: `rq1_text_light` untuk model berbasis embedding (misalnya BERTopic), dan `rq1_tokens` untuk model berbasis frekuensi kata (LDA/NMF). Dokumen yang layak adalah yang **relevan** (`is_relevant_candidate`), **layak pakai** (`common_is_usable`), dan **kemunculan pertama** dari suatu teks (agar teks berulang tidak menggembungkan topik). Kolom teks hanya diisi untuk dokumen kandidat.

## 3.1. Normalisasi Ejaan Tidak Baku

In [14]:
SLANG = {
    "yg": "yang", "yng": "yang", "dgn": "dengan", "dg": "dengan", "dng": "dengan", "utk": "untuk", "untk": "untuk",
    "tdk": "tidak", "gak": "tidak", "ga": "tidak", "gk": "tidak", "ngga": "tidak", "nggak": "tidak", "ngak": "tidak", "ngk": "tidak",
    "nga": "tidak", "ndak": "tidak", "enggak": "tidak", "engga": "tidak", "kaga": "tidak", "tak": "tidak",
    "udah": "sudah", "udh": "sudah", "sdh": "sudah", "sudh": "sudah", "blm": "belum", "belom": "belum",
    "klo": "kalau", "klu": "kalau", "kl": "kalau", "klau": "kalau", "klw": "kalau", "kalo": "kalau", "kalu": "kalau",
    "org": "orang", "orng": "orang", "jgn": "jangan", "jngn": "jangan", "tp": "tapi", "tpi": "tapi",
    "dr": "dari", "dri": "dari", "jd": "jadi", "jdi": "jadi", "krn": "karena", "karna": "karena", "krna": "karena",
    "tuh": "itu", "tu": "itu", "ni": "ini", "jg": "juga", "jga": "juga", "bener": "benar", "bnr": "benar",
    "gimana": "bagaimana", "gmn": "bagaimana", "gmna": "bagaimana", "gitu": "begitu", "gini": "begini",
    "pake": "pakai", "sampe": "sampai", "smpe": "sampai", "sy": "saya", "sya": "saya", "gw": "saya", "gue": "saya", "gua": "saya",
    "lu": "kamu", "lo": "kamu", "loe": "kamu", "lg": "lagi", "lgi": "lagi", "dpt": "dapat", "dapet": "dapat",
    "hrs": "harus", "dlm": "dalam", "byk": "banyak", "bnyk": "banyak", "bnyak": "banyak", "lbh": "lebih",
    "bs": "bisa", "trus": "terus", "trs": "terus", "koq": "kok", "pd": "pada", "aj": "saja", "aja": "saja", "ajah": "saja", "sj": "saja",
    "bgt": "banget", "bngt": "banget", "skrg": "sekarang", "skrng": "sekarang", "knp": "kenapa", "spt": "seperti", "kyk": "seperti",
    "kayak": "seperti", "bkn": "bukan", "mkn": "makan", "mkan": "makan", "tsb": "tersebut", "kpd": "kepada", "thn": "tahun",
    "dlu": "dulu", "emg": "memang", "mmg": "memang", "emang": "memang", "jln": "jalan", "brp": "berapa", "ntar": "nanti",
    "kemaren": "kemarin", "donk": "dong", "negri": "negeri", "mentri": "menteri", "progam": "program", "koropsi": "korupsi",
    "geratis": "gratis", "pinter": "pintar", "ortu": "orang tua", "ank": "anak", "bpk": "bapak", "jt": "juta", "rb": "ribu",
    "msh": "masih", "smua": "semua", "skolah": "sekolah", "tetep": "tetap", "denger": "dengar", "seneng": "senang",
    "ttg": "tentang", "tntg": "tentang", "sblm": "sebelum", "stlh": "setelah", "mrk": "mereka", "scr": "secara", "sbg": "sebagai",
    "kpn": "kapan", "dmn": "dimana", "bgmn": "bagaimana", "makasih": "terima kasih", "trims": "terima kasih", "mksh": "terima kasih",
    "embege": "mbg", "yaa": "ya", "yah": "ya", "nanya": "bertanya", "dll": "dan lain lain",
}
RE_SLANG = re.compile(r"\b(" + "|".join(sorted(map(re.escape, SLANG), key=len, reverse=True)) + r")\b")
RE_TAWA = [(re.compile(r"\b(?:ha){2,}h?\b"), "haha"), (re.compile(r"\b(?:wk){2,}w?k?\b"), "wkwk")]
RE_HURUF_ULANG = re.compile(r"([a-z])\1{2,}")  # 3 huruf berulang atau lebih menjadi 1 ("bangeeet" -> "banget")


def normalisasi_slang(teks):
    """Untuk teks yang sudah huruf kecil: ejaan tidak baku -> baku (daftar kecil dari kata tidak baku paling sering di korpus ini)."""
    teks = RE_HURUF_ULANG.sub(r"\1", teks)
    for pola, ganti in RE_TAWA:
        teks = pola.sub(ganti, teks)
    return RE_SLANG.sub(lambda m: SLANG[m.group(1)], teks)


def pecah_hashtag(m):
    """#KeracunanMBG -> 'Keracunan MBG' (pemisahan huruf besar harus dilakukan sebelum huruf kecil)."""
    return " " + re.sub(r"(?<=[a-z0-9])(?=[A-Z])|(?<=[A-Z])(?=[A-Z][a-z])", " ", m.group(1)) + " "


uji = ["Gak usah ngomong yg ga bener, bangeeet deh udh keracunan", "Dpr tdk peduli, krn org2 sdh kenyang wkwkwkwk", "#KeracunanMBG kok bs gt"]
pd.DataFrame({"asli": uji, "setelah normalisasi": [normalisasi_slang(RE_HASHTAG.sub(pecah_hashtag, t).lower()) for t in uji]})

,asli,setelah normalisasi
0,"Gak usah ngomong yg ga bener, bangeeet deh udh keracunan","tidak usah ngomong yang tidak benar, banget deh sudah keracunan"
1,"Dpr tdk peduli, krn org2 sdh kenyang wkwkwkwk","dpr tidak peduli, karena org2 sudah kenyang wkwk"
2,#KeracunanMBG kok bs gt,keracunan mbg kok bisa gt


Ejaan tidak baku dinormalisasi dengan kamus kecil (**147 entri**) yang disusun dari kata tidak baku paling sering muncul pada korpus ini (daftar kata di luar kamus akar kata Sastrawi). Isinya mencakup singkatan ("yg" menjadi "yang", "tdk" menjadi "tidak"), ejaan percakapan ("udah" menjadi "sudah"), dan seluruh variasi negasi ("gak", "ga", "nggak" menjadi "tidak"). Huruf yang berulang tiga kali atau lebih dikurangi menjadi satu ("bangeeet" menjadi "banget"), tawa diseragamkan ("wkwkwkwk" menjadi "wkwk"), dan hashtag camel-case dipecah ("#KeracunanMBG" menjadi "keracunan mbg"). Kamus ini **tidak lengkap**: kata tidak baku di luar daftar, nama panggilan, dan salah ketik tetap apa adanya.

## 3.2. Pembersihan Deskripsi Video

In [15]:
BOILER_BARIS = re.compile(
    r"^\W*(?:follow|like|subscribe|instagram|tiktok|twitter|facebook|spotify|website|youtube|telegram|vidio|produced by|artikel ini bisa|"
    r"selengkapnya|scan ulang|saksikan|ikuti|simak|update video|perbaharui|jangan lewatkan|yuk subscribe|sahabat kompastv|liputan ?6|sctv|"
    r"garuda ?tv|cnbc indonesia|terus ikuti|metro xtend|www\.|baca juga|video editor|produser|thumbnail|reporter|camera|cameraman|"
    r"narator|copyright|hak cipta|disclaimer|download|unduh|kunjungi)",
    re.I,
)
RE_BARIS_TAGAR = re.compile(r"^(?:\s*#\S+)+\s*$")
RE_BARIS_GARIS = re.compile(r"^[\s\-=_*~•·.]{3,}$")


def deskripsi_inti(teks_mentah):
    """Deskripsi video dari text_raw (baris baru masih ada): buang baris tautan sosmed, ajakan follow/subscribe, kredit, dan pemisah."""
    baris_ok = []
    for baris in teks_mentah.replace("\r", "").split("\n"):
        baris = baris.strip()
        if not baris or BOILER_BARIS.match(baris) or RE_BARIS_TAGAR.match(baris) or RE_BARIS_GARIS.match(baris):
            continue
        if RE_URL.fullmatch(baris):
            continue
        baris_ok.append(baris)
    return " ".join(baris_ok)


video = df[df["doc_type"] == "video"].sample(3, random_state=SEED)
for _, r in video.iterrows():
    print("SEBELUM:", len(r["text_clean"]), "karakter |", RE_MENTION.sub("@user", r["text_clean"][:160]), "...")
    print("SESUDAH:", len(deskripsi_inti(r["text_raw"])), "karakter |", RE_MENTION.sub("@user", deskripsi_inti(r["text_raw"])[:160]), "...\n")
panjang_sebelum = df.loc[df["doc_type"] == "video", "text_clean"].str.len().median()
panjang_sesudah = df.loc[df["doc_type"] == "video", "text_raw"].map(lambda s: len(deskripsi_inti(s))).median()
print(f"Median panjang deskripsi video: {panjang_sebelum:.0f} -> {panjang_sesudah:.0f} karakter setelah boilerplate dibuang")

SEBELUM: 1456 karakter | Purbaya Mau ‘Di-Noel-Kan’ hingga SPPG Tak Boleh Paksa MBG Menteri Keuangan Purbaya Yudhi Sadewa mengaku tak gentar mendengar peringatan Eks Wakil Menteri Ketena ...
SESUDAH: 641 karakter | Purbaya Mau ‘Di-Noel-Kan’ hingga SPPG Tak Boleh Paksa MBG Menteri Keuangan Purbaya Yudhi Sadewa mengaku tak gentar mendengar peringatan Eks Wakil Menteri Ketena ...

SEBELUM: 1020 karakter | TOP STORIES - Gempa Palu, Reformasi MBG, hingga Kesepakatan Damai AS-Iran MetroTV, Tiga peristiwa besar menjadi sorotan hari ini. Gempa bumi magnitudo 6,7 mengg ...
SESUDAH: 525 karakter | TOP STORIES - Gempa Palu, Reformasi MBG, hingga Kesepakatan Damai AS-Iran MetroTV, Tiga peristiwa besar menjadi sorotan hari ini. Gempa bumi magnitudo 6,7 mengg ...

SEBELUM: 820 karakter | [FULL] Kabar Merah Putih (21/05/2026) | tvOne Kabar Merah Putih, https://www.tvOnenews.com - [FULL] Kabar Merah Putih (21/05/2026) | tvOne Presiden Prabowo Subi ...
SESUDAH: 541 karakter | [FULL] Kabar Merah Putih (2

Deskripsi video memuat banyak **boilerplate saluran**: tautan sosial media, ajakan follow/subscribe, kredit produksi, dan pemisah. Jika dibiarkan, kata seperti "instagram", "subscribe", atau nama stasiun akan mendominasi topik. Baris-baris ini dibuang menggunakan `text_raw` (karena `text_clean` sudah menggabungkan baris), sehingga median panjang deskripsi turun dari **1.065 menjadi 588 karakter**. Iklan yang disisipkan di tengah teks (misalnya promosi produk) tidak terdeteksi oleh aturan baris ini dan masih bisa tersisa pada sebagian kecil video.

## 3.3. Teks Ringan (`rq1_text_light`)

In [16]:
def rq1_light(teks):
    """Teks ringan untuk model berbasis embedding (mis. BERTopic): huruf kecil, tanpa URL/mention/emoji, hashtag dipecah, ejaan baku."""
    teks = unicodedata.normalize("NFKC", html.unescape(teks))
    teks = RE_INVISIBLE.sub("", teks)
    teks = RE_URL.sub(" ", teks)
    teks = RE_MENTION.sub(" ", teks)
    teks = RE_HASHTAG.sub(pecah_hashtag, teks)
    teks = teks.lower()
    teks = emoji.replace_emoji(teks, " ")
    teks = normalisasi_slang(teks)
    teks = re.sub(r"[^\w\s.,!?'\"\-:;()%/]|_", " ", teks)
    return re.sub(r"\s+", " ", teks).strip()


# Tahap awal kelayakan RQ1: relevan, layak pakai, dan bukan pengulangan teks (hanya kemunculan pertama dipertahankan)
rq1_awal = df["is_relevant_candidate"] & df["common_is_usable"] & df["common_is_dup_first"]
print("Kandidat RQ1 sebelum filter token:", int(rq1_awal.sum()))
print(df[rq1_awal].groupby(["platform", "doc_type"]).size().to_string())

sumber = df["text_clean"].where(df["doc_type"] != "video", df["text_raw"].map(deskripsi_inti))  # deskripsi video memakai baris asli untuk membuang boilerplate
t0 = time.time()
df["rq1_text_light"] = pd.Series(np.nan, index=df.index, dtype="object")
df.loc[rq1_awal, "rq1_text_light"] = sumber[rq1_awal].map(rq1_light)
print(f"rq1_text_light dibuat dalam {time.time() - t0:.0f} detik")
contoh = df[rq1_awal & (df["doc_type"] != "video")].sample(4, random_state=SEED)
pd.DataFrame({"text_clean": aman(contoh["text_clean"], 85).values, "rq1_text_light": contoh["rq1_text_light"].str.slice(0, 85).values})

Kandidat RQ1 sebelum filter token: 97491
platform  doc_type
x         post         7103
          quote        1807
          reply        5499
youtube   comment     70035
          reply        8364
          video        4683
rq1_text_light dibuat dalam 24 detik


,text_clean,rq1_text_light
0,KOK PRABOWO DIAM SAAT BANYAK SISWA KERACUNAN MBG APA DIA TULI,kok prabowo diam saat banyak siswa keracunan mbg apa dia tuli
1,Jngn² sebelum dibagi disemprot racun tikus dulu untuk membunuh rakyat dan generasi ba,jngn2 sebelum dibagi disemprot racun tikus dulu untuk membunuh rakyat dan generasi ba
2,"gmn gak keracunan, tiap hari dikasih makan daging ayam, daging ayamnya rasanya mirip","bagaimana tidak keracunan, tiap hari dikasih makan daging ayam, daging ayamnya rasany"
3,"Bagus pak presiden lanjutkan,mbg kan membuka lapangan,terus lagi anak anak dapat gizi","bagus pak presiden lanjutkan,mbg kan membuka lapangan,terus lagi anak anak dapat gizi"


Kandidat RQ1 berjumlah **66.098 dokumen** (X 14.409; YouTube 51.689, termasuk 2.881 video). `rq1_text_light` dibuat dengan: NFKC, tanpa URL, mention, dan emoji, hashtag dipecah, huruf kecil, dan ejaan dibakukan. Tanda baca dipertahankan karena model embedding dapat memanfaatkannya, dan stemming tidak dilakukan.

## 3.4. Stopword dan Stemming (`rq1_tokens`)

In [17]:
stopword_sastrawi = set(StopWordRemoverFactory().get_stop_words())
STOP_TAMBAHAN = set("""nya sih dong deh nih tuh lah kok kan banget sekali pak bu bapak ibu ya yaa yah nah eh hmm
udah sudah aja juga lagi memang begitu begini benar kayak seperti gitu gini kamu saya aku kita kami dia mereka lu gw
tidak bukan belum jangan jadi lalu terus kemudian ada adalah ialah yaitu yakni dapat bisa akan mau harus hanya cuma masih
sama dengan dari untuk pada oleh atau serta tetapi tapi namun karena sebab agar supaya jika kalau bila apakah apa siapa kenapa
mengapa bagaimana dimana kapan berapa saat ketika sambil setiap semua para sebuah suatu satu dua tiga ini itu sini situ sana
dll dsb yg dgn utk tdk banyak lebih buat orang lain sangat bagi tanpa antara hingga""".split())
STOP_DOMAIN = {"mbg", "gratis", "gizi", "bergizi", "program", "makan", "embege"}  # topik umum yang ada di hampir semua dokumen; dibuang agar tidak mendominasi
STOP_SEMUA = stopword_sastrawi | STOP_TAMBAHAN | STOP_DOMAIN

stemmer = StemmerFactory().create_stemmer()
kamus_akar = stemmer.delegatedStemmer.dictionary
kamus_akar.words = set(kamus_akar.words)  # Sastrawi menyimpan kamus akar kata sebagai list (pencarian O(n)); set mempercepat stemming ratusan kali dengan hasil identik
cache_stem = {}


def stem(kata):
    hasil = cache_stem.get(kata)
    if hasil is None:
        hasil = cache_stem[kata] = stemmer.stem(kata)
    return hasil


def tokenisasi(teks):
    """Kata alfabet (>= 3 huruf) -> stopword dibuang -> stemming (Sastrawi, dengan cache) -> stopword domain dibuang lagi setelah stemming."""
    keluar = []
    for kata in re.findall(r"[a-z]{3,25}", teks):
        if kata in STOP_SEMUA:
            continue
        akar = stem(kata)
        if len(akar) >= 3 and akar not in STOP_SEMUA:
            keluar.append(akar)
    return " ".join(keluar)


kata_unik = {k for teks in df.loc[rq1_awal, "rq1_text_light"] for k in re.findall(r"[a-z]{3,25}", teks)} - STOP_SEMUA
print("Kosakata unik yang perlu di-stem:", len(kata_unik))
t0 = time.time()
for k in kata_unik:
    stem(k)
print(f"Stemming kosakata selesai dalam {time.time() - t0:.0f} detik (hasilnya disimpan di cache)")

df["rq1_tokens"] = pd.Series(np.nan, index=df.index, dtype="object")
df.loc[rq1_awal, "rq1_tokens"] = df.loc[rq1_awal, "rq1_text_light"].map(tokenisasi)
df["rq1_n_tokens"] = df["rq1_tokens"].str.count(r"\S+").astype("Int32")
print("Jumlah stopword:", len(STOP_SEMUA), f"(Sastrawi {len(stopword_sastrawi)}, tambahan {len(STOP_TAMBAHAN)}, domain {len(STOP_DOMAIN)})")
print("Contoh stemming:", {k: cache_stem[k] for k in ["keracunan", "beracun", "makanannya", "dimasak", "terkontaminasi", "kebersihan", "menjijikkan"] if k in cache_stem})
contoh = df[rq1_awal & (df["doc_type"] != "video")].sample(4, random_state=SEED + 1)
pd.DataFrame({"rq1_text_light": contoh["rq1_text_light"].str.slice(0, 75).values, "rq1_tokens": contoh["rq1_tokens"].str.slice(0, 75).values})

Kosakata unik yang perlu di-stem: 73584
Stemming kosakata selesai dalam 12 detik (hasilnya disimpan di cache)
Jumlah stopword: 197 (Sastrawi 123, tambahan 123, domain 7)
Contoh stemming: {'keracunan': 'racun', 'beracun': 'racun', 'makanannya': 'makan', 'dimasak': 'masak', 'terkontaminasi': 'kontaminasi', 'kebersihan': 'bersih', 'menjijikkan': 'jijik'}


,rq1_text_light,rq1_tokens
0,makan bergizi gratis beruba menjadi makan beracun gratis,uba racun
1,astaghfirullahaladzim punya pemimpin sudah buta hatinya?! masih bertanya sa,astaghfirullahaladzim punya pimpin buta hati tanya salah padahal racun koru
2,cb ks mbg yang di makan anak2 yang keracunan mau tidak si bowo,anak racun bowo
3,sudah banyak yanh keracunan mengapa masih diterima,yanh racun terima


`rq1_tokens` dibuat dari `rq1_text_light`: kata alfabet (minimal 3 huruf), stopword dibuang, di-stem dengan **Sastrawi**, lalu stopword domain dibuang lagi setelah stemming. Daftar stopword berjumlah 197 kata (Sastrawi, tambahan percakapan dan kata sambung, serta domain: `mbg`, `gratis`, `gizi`, `program`, `makan`) karena kata domain ada di hampir semua dokumen dan tidak membedakan topik. Stemming memproses **59.987 kosakata unik** dan hasilnya disimpan di cache. Kamus akar kata Sastrawi disimpan sebagai `list` sehingga pencarian lambat; mengubahnya menjadi `set` mempercepat proses sekitar 700 kali dengan hasil identik (diuji pada 1.500 kata).

Stemming bersifat agresif dan bisa menghasilkan akar yang kurang natural (misalnya "pelaku" menjadi "laku", "pejabat" menjadi "jabat"). Untuk model berbasis embedding sebaiknya memakai `rq1_text_light`, bukan `rq1_tokens`.

## 3.5. Kategori Isu dan Kelayakan RQ1

In [18]:
ISU = {
    "keracunan": r"racun",
    "basi_busuk": r"basi|busuk|\bbau\b|berbau|lendir|tidak layak|layak konsumsi|kedaluwarsa|kadaluarsa|kadaluwarsa",
    "kontaminasi_bakteri": r"bakteri|kontaminasi|salmonella|e\.? ?coli",
    "benda_asing": r"belatung|\bulat\b|kecoa|serangga|benda asing|rambut|plastik",
    "higiene_sanitasi": r"higien|kebersihan|sanitasi|kotor|jorok|steril",
    "kualitas_matang": r"mentah|hambar|tidak matang|kurang matang|belum matang",
    "gejala_kesehatan": r"muntah|\bmual\b|diare|sakit perut|rawat inap|opname|alergi",
}
RE_ISU = {nama: re.compile(pola) for nama, pola in ISU.items()}


def kategori_isu(teks):
    """Kategori masalah yang kata kuncinya muncul (label lemah untuk memeriksa topik yang ditemukan model, bukan label sebenarnya)."""
    ada = [nama for nama, pola in RE_ISU.items() if pola.search(teks)]
    return ";".join(ada) if ada else np.nan


df["rq1_issue_categories"] = pd.Series(np.nan, index=df.index, dtype="object")
df.loc[rq1_awal, "rq1_issue_categories"] = df.loc[rq1_awal, "rq1_text_light"].map(kategori_isu)
df["rq1_eligible"] = rq1_awal & (df["rq1_n_tokens"].fillna(0) >= MIN_RQ1_TOKENS)

print(f"Kandidat RQ1: {int(rq1_awal.sum())} -> rq1_eligible (>= {MIN_RQ1_TOKENS} token): {int(df['rq1_eligible'].sum())}")
print(df[df["rq1_eligible"]].groupby(["platform", "doc_type"]).size().to_string())
e = df[df["rq1_eligible"]]
print("\nJumlah token per dokumen (median, rata-rata):")
print(e.groupby(["platform", "doc_type"])["rq1_n_tokens"].agg(["median", "mean"]).astype(float).round(1).to_string())
print("\nKategori isu (satu dokumen bisa memuat lebih dari satu):")
kat = e["rq1_issue_categories"].str.split(";").explode()
print(pd.DataFrame({"jumlah": kat.value_counts(), "persen": (kat.value_counts() / len(e) * 100).round(1)}).to_string())
print(f"Dokumen eligible tanpa kategori isu: {int(e['rq1_issue_categories'].isna().sum())}")
df_token = Counter(t for toks in e["rq1_tokens"] for t in set(toks.split()))
print("\nToken dengan frekuensi dokumen tertinggi (jumlah dokumen, persen):")
print({t: f"{c} ({c / len(e):.0%})" for t, c in df_token.most_common(15)})
print("Ukuran kosakata:", len(df_token), "| token yang muncul di >= 5 dokumen:", sum(c >= 5 for c in df_token.values()))
catat("3.5 RQ1", "rq1_text_light, rq1_tokens, rq1_n_tokens, rq1_issue_categories, rq1_eligible", int(rq1_awal.sum()), int(df["rq1_eligible"].sum()))

Kandidat RQ1: 97491 -> rq1_eligible (>= 3 token): 93633
platform  doc_type
x         post         7054
          quote        1767
          reply        5395
youtube   comment     66727
          reply        8008
          video        4682

Jumlah token per dokumen (median, rata-rata):
                   median  mean
platform doc_type              
x        post        19.0  39.8
         quote       19.0  32.9
         reply       17.0  19.4
youtube  comment     10.0  15.4
         reply       11.0  17.3
         video       62.0  73.3

Kategori isu (satu dokumen bisa memuat lebih dari satu):
                      jumlah  persen
rq1_issue_categories                
keracunan              66524    71.0
basi_busuk             17248    18.4
higiene_sanitasi        6750     7.2
benda_asing             5550     5.9
gejala_kesehatan        4102     4.4
kualitas_matang         3324     3.6
kontaminasi_bakteri     2094     2.2
Dokumen eligible tanpa kategori isu: 506

Token dengan frekuens

Sebanyak **63.375 dokumen** layak untuk RQ1 (`rq1_eligible`): X 14.216 (7.054 post, 1.767 quote, 5.395 reply) dan YouTube 49.159 (41.173 komentar, 5.106 reply, 2.880 video). Dokumen dengan kurang dari 3 token dikeluarkan (2.723 dokumen).

`rq1_issue_categories` adalah **label lemah** berbasis kata kunci yang diturunkan dari taksonomi query pengumpulan, dipakai untuk memeriksa apakah topik yang ditemukan model sejalan dengan jenis masalah. Distribusinya: keracunan **70,0%**, basi/busuk 18,9%, higiene/sanitasi 7,3%, benda asing 6,7%, kualitas/kematangan 4,3%, gejala kesehatan 4,2%, dan kontaminasi bakteri 2,5% (satu dokumen bisa memuat beberapa kategori). Dominasi kategori keracunan mencerminkan kata kunci yang dipakai saat pengumpulan, sehingga model topik perlu diuji agar tidak hanya menemukan "keracunan" berulang kali. Token `racun` ada di 68% dokumen, sedangkan kosakata mencapai 45.197 token (10.076 token muncul di minimal 5 dokumen).

# 4. Pre-Processing RQ2: Lokasi (kolom `rq2_`)

RQ2 memerlukan satu **provinsi** per dokumen. Provinsi diambil dari lokasi yang disebut pada teks (`province` dari merge), dibakukan, dan dikelompokkan per pulau. Karena hanya sedikit komentar YouTube yang menyebut lokasi sendiri, lokasi boleh **diwarisi dari konteks** (video) dengan syarat ketat dan disimpan di kolom terpisah agar dapat dibandingkan dengan dan tanpa pewarisan.

## 4.1. Normalisasi Provinsi

In [19]:
PROVINSI_PULAU = {
    "Aceh": "Sumatera", "Sumatera Utara": "Sumatera", "Sumatera Barat": "Sumatera", "Riau": "Sumatera", "Kepulauan Riau": "Sumatera",
    "Jambi": "Sumatera", "Sumatera Selatan": "Sumatera", "Kepulauan Bangka Belitung": "Sumatera", "Bengkulu": "Sumatera", "Lampung": "Sumatera",
    "DKI Jakarta": "Jawa", "Banten": "Jawa", "Jawa Barat": "Jawa", "Jawa Tengah": "Jawa", "Daerah Istimewa Yogyakarta": "Jawa", "Jawa Timur": "Jawa",
    "Bali": "Bali dan Nusa Tenggara", "Nusa Tenggara Barat": "Bali dan Nusa Tenggara", "Nusa Tenggara Timur": "Bali dan Nusa Tenggara",
    "Kalimantan Barat": "Kalimantan", "Kalimantan Tengah": "Kalimantan", "Kalimantan Selatan": "Kalimantan", "Kalimantan Timur": "Kalimantan", "Kalimantan Utara": "Kalimantan",
    "Sulawesi Utara": "Sulawesi", "Sulawesi Tengah": "Sulawesi", "Sulawesi Selatan": "Sulawesi", "Sulawesi Tenggara": "Sulawesi", "Gorontalo": "Sulawesi", "Sulawesi Barat": "Sulawesi",
    "Maluku": "Maluku", "Maluku Utara": "Maluku",
    "Papua": "Papua", "Papua Barat": "Papua", "Papua Barat Daya": "Papua", "Papua Selatan": "Papua", "Papua Tengah": "Papua", "Papua Pegunungan": "Papua",
}


def daftar_provinsi(nilai):
    return [p.strip() for p in nilai.split(";")] if isinstance(nilai, str) else []


plist = df["province"].map(daftar_provinsi)
semua = Counter(p for lst in plist for p in lst)
tak_dikenal = sorted(set(semua) - set(PROVINSI_PULAU))
print(f"Nama provinsi unik pada data: {len(semua)} | tidak ada dalam daftar 38 provinsi: {tak_dikenal or 'tidak ada'}")
assert not tak_dikenal, "Ada nama provinsi yang belum dibakukan"

df["rq2_n_provinces"] = plist.map(len).astype("int8")
df["rq2_province_list"] = plist.map("; ".join).replace("", np.nan)
df["_prov_teks"] = plist.map(lambda l: l[0] if len(l) == 1 else np.nan)  # provinsi tunggal dari teks dokumen itu sendiri
print("\nJumlah provinsi yang disebut per dokumen (dokumen ber-lokasi):")
print(df[df["rq2_n_provinces"] > 0].groupby("platform")["rq2_n_provinces"].value_counts().unstack(fill_value=0).iloc[:, :6].to_string())
print("\nCakupan lokasi pada teks sendiri (persen dokumen), per platform dan jenis dokumen:")
print((df.groupby(["platform", "doc_type"])["rq2_n_provinces"].apply(lambda s: (s > 0).mean()) * 100).round(1).to_string())
print("Dokumen dengan nama lokasi tanpa provinsi (nama ambigu, mis. Banjar):", int((df["location_names"].notna() & df["province"].isna()).sum()))
catat("4.1 Normalisasi provinsi", "rq2_province_list, rq2_n_provinces (38 provinsi baku; format dipisah ';')", len(df), int((df["rq2_n_provinces"] > 0).sum()))

Nama provinsi unik pada data: 38 | tidak ada dalam daftar 38 provinsi: tidak ada

Jumlah provinsi yang disebut per dokumen (dokumen ber-lokasi):
rq2_n_provinces      1     2    3   4   5   6
platform                                     
x                 5074   294   97  39   9   6
youtube          26810  1662  337  97  29  23

Cakupan lokasi pada teks sendiri (persen dokumen), per platform dan jenis dokumen:
platform  doc_type
x         post        35.1
          quote       11.1
          reply        6.1
youtube   comment      1.9
          reply        1.7
          video       51.7
Dokumen dengan nama lokasi tanpa provinsi (nama ambigu, mis. Banjar): 0


Seluruh nama provinsi pada data sudah baku (38 provinsi, termasuk enam provinsi Papua hasil pemekaran; "DI Yogyakarta" sudah diganti "Daerah Istimewa Yogyakarta" oleh merge), dan tidak ada nama yang belum dikenal. Kolom yang dibuat: `rq2_province_list` (semua provinsi pada teks sendiri), `rq2_n_provinces`, dan provinsi tunggal. Sebanyak 445 dokumen X dan 1.275 dokumen YouTube menyebut lebih dari satu provinsi (misalnya daftar kasus keracunan di banyak daerah), dan dokumen seperti ini tidak diberi provinsi tunggal.

Cakupan lokasi pada teks sendiri **sangat timpang**: 35,1% pada post X dan 49,4% pada deskripsi video, tetapi hanya **1,9% pada komentar dan 1,7% pada reply YouTube**, serta 6,1% pada reply X. Hal ini menjadi alasan perlunya pewarisan konteks.

## 4.2. Pewarisan Lokasi dari Konteks

In [20]:
# Warisi lokasi dari konteks (video untuk YouTube, unggahan akar untuk X) hanya jika: dokumen tidak menyebut provinsi sendiri,
# dan konteksnya menyebut tepat SATU provinsi. Hasilnya disimpan di kolom terpisah dan tidak menimpa 'province'.
prov_konteks = df["context_id"].map(pd.Series(df["_prov_teks"].to_numpy(), index=df["doc_id"]))
punya_konteks = (df["context_id"] != df["doc_id"])
df["rq2_province_context"] = prov_konteks.where(punya_konteks & (df["rq2_n_provinces"] == 0))
df["rq2_location_source"] = np.select(
    [df["rq2_n_provinces"] > 0, df["rq2_province_context"].notna()], ["text", "context"], default="none"
)
df["rq2_province"] = df["_prov_teks"].fillna(df["rq2_province_context"])  # satu provinsi (teks sendiri, atau konteks); kosong jika tidak ada atau lebih dari satu
df["rq2_island_group"] = df["rq2_province"].map(PROVINSI_PULAU)

print("Sumber lokasi (semua dokumen):")
print(pd.crosstab(df["platform"], df["rq2_location_source"]).to_string())
relevan = df["is_relevant_candidate"] & (df["doc_type"] != "video")
print("\nDi antara dokumen relevan non-video, persen yang memiliki informasi provinsi:")
dasar = df[relevan].groupby("platform").size()
print(pd.DataFrame({
    "teks sendiri (>= 1 provinsi)": (df[relevan & (df["rq2_n_provinces"] > 0)].groupby("platform").size() / dasar * 100).round(1),
    "teks sendiri (1 provinsi)": (df[relevan & df["_prov_teks"].notna()].groupby("platform").size() / dasar * 100).round(1),
    "rq2_province (teks atau konteks)": (df[relevan & df["rq2_province"].notna()].groupby("platform").size() / dasar * 100).round(1),
}).to_string())
print("\nDokumen dengan > 1 provinsi (tidak diwarisi dan rq2_province kosong):", int((df["rq2_n_provinces"] > 1).sum()))
catat("4.2 Pewarisan lokasi dari konteks", "rq2_province_context, rq2_location_source, rq2_province, rq2_island_group", len(df), int(df["rq2_province"].notna().sum()))

Sumber lokasi (semua dokumen):
rq2_location_source  context    none   text
platform                                   
x                        253   18611   5671
youtube               480549  575818  28986

Di antara dokumen relevan non-video, persen yang memiliki informasi provinsi:
          teks sendiri (>= 1 provinsi)  teks sendiri (1 provinsi)  rq2_province (teks atau konteks)
platform                                                                                           
x                                 24.0                       20.9                              22.2
youtube                            2.5                        2.2                              58.5

Dokumen dengan > 1 provinsi (tidak diwarisi dan rq2_province kosong): 2773


Pewarisan hanya dilakukan jika (1) dokumen tidak menyebut provinsi sendiri dan (2) konteksnya (video atau unggahan akar) menyebut **tepat satu** provinsi. Hasilnya disimpan di `rq2_province_context`, dengan `rq2_location_source` (`text`, `context`, `none`) dan `rq2_province` sebagai provinsi tunggal final. Dokumen yang menyebut lebih dari satu provinsi (1.887 dokumen) tidak diwarisi dan tidak diberi `rq2_province`.

Dampaknya besar pada YouTube: di antara komentar/reply relevan, yang memiliki provinsi naik dari **2,4% (teks sendiri) menjadi 59,2%** setelah pewarisan; pada X hanya 22,2% karena konteks X jarang tersedia (253 dokumen mewarisi lokasi). Perlu diingat bahwa **pewarisan adalah asumsi**: komentar di bawah video tentang kasus di satu provinsi belum tentu membahas provinsi itu.

## 4.3. Kelayakan RQ2

In [21]:
df["rq2_eligible"] = df["rq1_eligible"] & df["rq2_province"].notna()
df["rq2_eligible_text_only"] = df["rq2_eligible"] & (df["rq2_location_source"] == "text")
print(f"rq2_eligible (RQ1 eligible dan punya satu provinsi): {int(df['rq2_eligible'].sum())} | hanya dari teks sendiri: {int(df['rq2_eligible_text_only'].sum())}")
e2 = df[df["rq2_eligible"]]
tabel = pd.crosstab(e2["rq2_province"], e2["platform"])
tabel["total"] = tabel.sum(axis=1)
tabel["teks_sendiri"] = e2[e2["rq2_location_source"] == "text"].groupby("rq2_province").size().reindex(tabel.index).fillna(0).astype(int)
warisan = e2[e2["rq2_location_source"] == "context"]
tabel["konteks_unik_warisan"] = warisan.groupby("rq2_province")["context_id"].nunique().reindex(tabel.index).fillna(0).astype(int)
terbesar = warisan.groupby("rq2_province")["context_id"].agg(lambda s: s.value_counts(normalize=True).iloc[0] * 100).reindex(tabel.index).round(0)
tabel["% warisan dari 1 video"] = terbesar
tabel["pulau"] = tabel.index.map(PROVINSI_PULAU)
tabel["jarang (< 30 dok)"] = tabel["total"] < 30
print("\nDokumen RQ2 per provinsi (diurutkan). konteks_unik_warisan = jumlah video berbeda di balik dokumen yang lokasinya diwarisi:")
print(tabel.sort_values("total", ascending=False).to_string())
print("\nPer kelompok pulau:")
print(pd.crosstab(e2["rq2_island_group"], e2["platform"]).assign(total=lambda t: t.sum(axis=1)).sort_values("total", ascending=False).to_string())
print("\nBandingkan dengan merge (relevant_docs_per_province.csv): total dokumen relevan ber-provinsi pada teks sendiri =", int(((df["rq2_location_source"] == "text") & df["is_relevant_candidate"]).sum()))
catat("4.3 Kelayakan RQ2", "rq2_eligible, rq2_eligible_text_only", int(df["rq1_eligible"].sum()), int(df["rq2_eligible"].sum()))

rq2_eligible (RQ1 eligible dan punya satu provinsi): 49131 | hanya dari teks sendiri: 7386

Dokumen RQ2 per provinsi (diurutkan). konteks_unik_warisan = jumlah video berbeda di balik dokumen yang lokasinya diwarisi:
platform                      x  youtube  total  teks_sendiri  konteks_unik_warisan  % warisan dari 1 video                   pulau  jarang (< 30 dok)
rq2_province                                                                                                                                          
Jawa Barat                  555    13845  14400          1623                   687                    33.0                    Jawa              False
Jawa Tengah                 641     7191   7832          1275                   444                     4.0                    Jawa              False
DKI Jakarta                 235     7319   7554           714                  1021                     3.0                    Jawa              False
Jawa Timur                  4

`rq2_eligible` (dokumen layak RQ1 yang memiliki provinsi tunggal) berjumlah **31.853 dokumen**, tetapi hanya **5.632 (17,7%)** yang lokasinya berasal dari teks sendiri (`rq2_eligible_text_only`); sisanya 82% bergantung pada pewarisan. Ini menjadi catatan penting bagi analisis RQ2, dengan beberapa temuan:

* **Dominasi Pulau Jawa:** 25.750 dokumen (81%), dengan Jawa Barat saja 11.640 (37%).
* **Pewarisan terkonsentrasi pada sedikit video.** Kolom `% warisan dari 1 video` menunjukkan, misalnya, 81% dokumen berlokasi Banten, 87% Lampung, dan 72% Sumatera Selatan berasal dari **satu video yang sama**, dan 40% Jawa Barat dari satu video. Satu video viral dapat membuat sebuah provinsi tampak besar, sehingga hasil per provinsi perlu dilaporkan bersama jumlah video unik atau dengan batas kontribusi per video.
* **12 provinsi punya kurang dari 30 dokumen** (kolom `jarang`; mayoritas provinsi Papua, Kalimantan, dan Maluku Utara). Disarankan menggabungkannya per kelompok pulau (`rq2_island_group`) atau memberi ambang minimum.
* Sebagai pembanding, tabel merge (`relevant_docs_per_province.csv`) menghitung 6.545 dokumen relevan berlokasi dari teks sendiri; angka ini lebih besar daripada 5.632 karena `rq2_eligible` juga mensyaratkan dokumen kemunculan pertama, bukan spam, dan provinsi tunggal.

Disarankan melaporkan RQ2 **dengan dan tanpa** pewarisan (`rq2_eligible` dan `rq2_eligible_text_only`). Perbandingan antar provinsi sebaiknya memakai proporsi terhadap jumlah dokumen provinsi tersebut, bukan hitungan mentah, karena cakupan waktu dan platform tidak merata.

# 5. Pre-Processing RQ3: Sentiment Analysis (kolom `rq3_`)

Untuk RQ3 disiapkan **teks minimal** yang mempertahankan sinyal sentimen: negasi ("tidak", "bukan"), emoji, tanda seru, dan kata sambung dipertahankan; **tidak ada stopword removal dan tidak ada stemming**, karena model sentimen berbasis transformer (misalnya IndoBERT) bekerja pada kalimat utuh. URL menjadi `<URL>` dan mention menjadi `@user`. Nama emoji versi bahasa Indonesia disediakan di kolom terpisah untuk metode berbasis kamus kata.

Kelayakan RQ3 dibagi dua tingkat. **Tingkat A (utama):** dokumen layak pakai, bukan video, dan relevan (menyebut keamanan pangan sendiri). **Tingkat B (sensitivitas):** dokumen layak pakai tanpa kata kunci pangan sendiri, tetapi video atau konteksnya membahas keamanan pangan. Video tidak dimasukkan karena deskripsinya adalah teks berita, bukan opini publik.

## 5.1. Teks Minimal untuk Sentimen (`rq3_text`)

In [22]:
RE_WAKTU_VIDEO = re.compile(r"\b\d{1,2}:\d{2}(?::\d{2})?\b")  # penanda waktu video YouTube ("2:34")
RE_MENTION_BERUNTUN = re.compile(r"(?:@user\s*){2,}")


def rq3_teks(teks):
    """Teks minimal untuk analisis sentimen: negasi, emoji, tanda seru, dan kata sambung dipertahankan; tanpa stopword dan tanpa stemming."""
    teks = unicodedata.normalize("NFKC", html.unescape(teks))
    teks = re.sub("[\u00ad\u200b\u200c\u200e\u200f\u202a-\u202e\u2060-\u2064\ufeff]", "", teks)  # ZWJ (U+200D) dipertahankan agar emoji gabungan utuh
    teks = RE_HASHTAG.sub(pecah_hashtag, teks)
    teks = RE_WAKTU_VIDEO.sub(" ", teks)
    teks = teks.lower()
    teks = RE_URL.sub(" <URL> ", teks)
    teks = RE_MENTION.sub(" @user ", teks)
    teks = RE_MENTION_BERUNTUN.sub("@user ", teks)
    teks = normalisasi_slang(teks)
    teks = re.sub(r"([!?.,])\1{3,}", r"\1\1\1", teks)
    return re.sub(r"\s+", " ", teks).strip()


def emoji_ke_teks(teks):
    """Emoji -> namanya dalam bahasa Indonesia (untuk metode berbasis kamus kata); emoji pada teks asli tidak diubah."""
    d = emoji.demojize(teks, delimiters=("⟦", "⟧"), language="id")
    return re.sub(r"\s+", " ", re.sub(r"⟦([^⟧]+)⟧", lambda m: " " + m.group(1).replace("_", " ") + " ", d)).strip()


uji = ["@abc @def Ya Allah anak2 keracunan lagi!!!!! 😭😭 gak tau mau ngomong apa, https://x.com/a/status/1 #KeracunanMBG", "2:34 bangeeet enak makanannya 👍🏽 tdk ada yg basi"]
pd.DataFrame({"asli": uji, "rq3_text": [rq3_teks(t) for t in uji], "rq3_text_emoji": [emoji_ke_teks(rq3_teks(t)) for t in uji]})

,asli,rq3_text,rq3_text_emoji
0,"@abc @def Ya Allah anak2 keracunan lagi!!!!! 😭😭 gak tau mau ngomong apa, https://x.com/a/status/1 #Keracun...","@user ya allah anak2 keracunan lagi!!! 😭😭 tidak tau mau ngomong apa, <URL> keracunan mbg",@user ya allah anak2 keracunan lagi!!! wajah menangis keras wajah menangis keras tidak tau mau ngomong apa...
1,2:34 bangeeet enak makanannya 👍🏽 tdk ada yg basi,banget enak makanannya 👍🏽 tidak ada yang basi,banget enak makanannya jempol ke atas warna kulit sedang tidak ada yang basi


Contoh uji memperlihatkan perilaku fungsi: dua mention beruntun menjadi satu `@user`, tautan menjadi `<URL>`, hashtag dipecah, penanda waktu video ("2:34") dibuang, huruf berulang dan ejaan tidak baku dibakukan ("gak" menjadi "tidak" sehingga negasi tetap terbaca), tanda seru berulang dibatasi tiga, dan emoji dipertahankan (lalu dapat diubah menjadi nama Indonesia pada `rq3_text_emoji_id`).

## 5.2. Tingkat Kelayakan dan Pembuatan Teks RQ3

In [23]:
bukan_video = df["doc_type"] != "video"
tier_a = df["common_is_usable"] & df["is_relevant_candidate"] & bukan_video
tier_b = df["common_is_usable"] & ~df["is_relevant_candidate"] & bukan_video & df["common_context_mentions_food_safety"] & df["context_mentions_mbg"]
df["rq3_tier"] = np.select([tier_a, tier_b], ["A", "B"], default="")
df["rq3_tier"] = df["rq3_tier"].replace("", np.nan)
df["rq3_eligible"] = tier_a
df["rq3_eligible_ext"] = tier_a | tier_b
print("Tingkat A (utama): menyebut keamanan pangan sendiri dan relevan dengan MBG | Tingkat B (sensitivitas): tanpa kata kunci sendiri, tetapi konteksnya membahas keamanan pangan")
print(pd.crosstab([df["platform"], df["doc_type"]], df["rq3_tier"].fillna("-")).to_string())
print("\nVideo (deskripsi berita) tidak dimasukkan ke RQ3 karena bukan opini publik; duplikat teks dari penulis berbeda tetap dipertahankan (bobot: common_dup_group_size).")

t0 = time.time()
ada = df["rq3_eligible_ext"]
df["rq3_text"] = pd.Series(np.nan, index=df.index, dtype="object")
df.loc[ada, "rq3_text"] = df.loc[ada, "text_clean"].map(rq3_teks)
print(f"rq3_text dibuat untuk {int(ada.sum())} dokumen dalam {time.time() - t0:.0f} detik")

df["rq3_n_emoji"] = pd.Series(pd.NA, index=df.index, dtype="Int32")
df.loc[ada, "rq3_n_emoji"] = df.loc[ada, "text_clean"].map(emoji.emoji_count).astype("Int32")
df["rq3_text_emoji_id"] = pd.Series(np.nan, index=df.index, dtype="object")
punya_emoji = ada & (df["rq3_n_emoji"].fillna(0) > 0)
df.loc[ada, "rq3_text_emoji_id"] = df.loc[ada, "rq3_text"]
df.loc[punya_emoji, "rq3_text_emoji_id"] = df.loc[punya_emoji, "rq3_text"].map(emoji_ke_teks)
a = df[df["rq3_eligible"]]
print(f"Dokumen tingkat A mengandung emoji: {(a['rq3_n_emoji'] > 0).mean():.1%} | mengandung kata negasi 'tidak': {a['rq3_text'].str.contains(r'\btidak\b').mean():.1%} | mengandung '!': {a['rq3_text'].str.contains('!').mean():.1%}")
print("Panjang rq3_text (kata), median per platform:", a.groupby("platform")["rq3_text"].apply(lambda s: s.str.count(r"\S+").median()).to_dict())
contoh = a[a["rq3_n_emoji"] > 0].sample(3, random_state=SEED)
pd.DataFrame({"text_clean": aman(contoh["text_clean"], 80).values, "rq3_text_emoji_id": contoh["rq3_text_emoji_id"].str.slice(0, 100).values})

Tingkat A (utama): menyebut keamanan pangan sendiri dan relevan dengan MBG | Tingkat B (sensitivitas): tanpa kata kunci sendiri, tetapi konteksnya membahas keamanan pangan
rq3_tier                -      A       B
platform doc_type                       
x        post        6815   7170       0
         quote        664   1817       0
         reply       2464   5521      84
youtube  comment   667304  71958  152879
         reply     139293   8470   27786
         video      17663      0       0

Video (deskripsi berita) tidak dimasukkan ke RQ3 karena bukan opini publik; duplikat teks dari penulis berbeda tetap dipertahankan (bobot: common_dup_group_size).
rq3_text dibuat untuk 275685 dokumen dalam 14 detik
Dokumen tingkat A mengandung emoji: 17.9% | mengandung kata negasi 'tidak': 38.7% | mengandung '!': 6.2%
Panjang rq3_text (kata), median per platform: {'x': 32.0, 'youtube': 18.0}


,text_clean,rq3_text_emoji_id
0,Generasii DIRACUN MASSAL.MENGERIKAN😮😮😮😮😮,generasii diracun massal.mengerikan wajah dengan mulut terbuka wajah dengan mulut terbuka wajah deng
1,Emang semua kacang panjang ada ulatnya? Ibu gw masak sup seumur hidup gak pernah,memang semua kacang panjang ada ulatnya? ibu saya masak sup seumur hidup tidak pernah keluar itu hew
2,Hujat provokasi trus.. Smpe hati busuk 😂😂😂😂😂,hujat provokasi terus.. sampai hati busuk wajah gembira berurai air mata wajah gembira berurai air m


Tingkat A berjumlah **64.735 dokumen**: X 14.508 dan YouTube 50.227. Tingkat B menambah **114.710 dokumen** (hampir seluruhnya komentar dan reply YouTube di bawah video yang membahas keamanan pangan), sehingga `rq3_eligible_ext` berjumlah 179.445. Tingkat B **jauh lebih berisik**: kebanyakan komentar di sana membahas politik atau anggaran, bukan keamanan pangan, sehingga sebaiknya hanya dipakai sebagai uji sensitivitas.

Pada tingkat A, 18,5% dokumen memuat emoji, 38,7% memuat kata negasi "tidak", dan 6,5% memuat tanda seru; ketiganya terjaga di `rq3_text`. Median panjang teks adalah 32 kata pada X dan 18 kata pada YouTube. Duplikat teks dari penulis berbeda tetap dipertahankan dan dapat diberi bobot dengan `common_dup_group_size`. Sentimen negatif belum tentu sama dengan kekhawatiran pangan, karena banyak komentar negatif membahas hal lain; pada tahap analisis, `mentions_food_safety` atau `rq1_issue_categories` dapat dipakai untuk memisahkannya.

## 5.3. Sampel Pelabelan Manual

In [24]:
# Sampel pelabelan manual (tingkat A): acak per platform, untuk mengukur ketepatan model/leksikon sentimen pada data MBG
label_idx = df[df["rq3_eligible"]].groupby("platform", group_keys=False).sample(LABEL_SAMPLE_PER_PLATFORM, random_state=SEED).index
df["rq3_label_sample"] = df.index.isin(label_idx)
print("Sampel pelabelan:", int(df["rq3_label_sample"].sum()), "dokumen")
print(pd.crosstab(df.loc[df["rq3_label_sample"], "platform"], df.loc[df["rq3_label_sample"], "doc_type"]).to_string())
print("Sebaran waktu sampel (persen per kuartal, UTC):")
kuartal = pd.to_datetime(df.loc[df["rq3_label_sample"], "published_at_utc"], utc=True).dt.tz_localize(None).dt.to_period("Q").astype(str)
print((pd.crosstab(kuartal, df.loc[df["rq3_label_sample"], "platform"], normalize="columns") * 100).round(1).to_string())
catat("5 RQ3", "rq3_tier (A/B), rq3_eligible, rq3_eligible_ext, rq3_text, rq3_text_emoji_id, rq3_n_emoji, rq3_label_sample", int(df["rq3_eligible_ext"].sum()), int(df["rq3_eligible"].sum()))

Sampel pelabelan: 800 dokumen
doc_type  comment  post  quote  reply
platform                             
x               0   192     41    167
youtube       350     0      0     50
Sebaran waktu sampel (persen per kuartal, UTC):
platform             x  youtube
published_at_utc               
2025Q1             4.2      6.0
2025Q2             5.2      2.8
2025Q3            18.0     20.2
2025Q4            13.8     10.8
2026Q1            25.2     11.5
2026Q2            15.8     11.2
2026Q3            17.8     37.0
2026Q4             0.0      0.5


Sampel pelabelan manual berisi **800 dokumen tingkat A** (400 per platform, acak dengan benih tetap): X terdiri dari 192 post, 41 quote, dan 167 reply; YouTube terdiri dari 356 komentar dan 44 reply. Sebaran per kuartal mengikuti populasi sehingga ikut tidak merata (YouTube hanya 1,5% pada kuartal II 2025). Sampel ini dimaksudkan untuk mengukur ketepatan model atau kamus sentimen pada domain MBG sebelum hasilnya dipercaya, sekaligus sebagai bahan memeriksa kategori sentimen (positif, negatif, netral) dan apakah dokumen benar-benar terkait keamanan pangan.

# 6. Penggabungan, Validasi, dan Penyimpanan

## 6.1. Penyusunan Dataset Processed

In [25]:
KOLOM_UMUM = [c for c in df.columns if c.startswith("common_")]
KOLOM_RQ1 = [c for c in df.columns if c.startswith("rq1_")]
KOLOM_RQ2 = [c for c in df.columns if c.startswith("rq2_")]
KOLOM_RQ3 = [c for c in df.columns if c.startswith("rq3_")]
df = df.drop(columns=[c for c in df.columns if c.startswith("_")])  # kolom bantu (bentuk pembanding)
out = df[ORIG_COLS + KOLOM_UMUM + KOLOM_RQ1 + KOLOM_RQ2 + KOLOM_RQ3].copy()
print(f"Kolom: asli {len(ORIG_COLS)} + umum {len(KOLOM_UMUM)} + RQ1 {len(KOLOM_RQ1)} + RQ2 {len(KOLOM_RQ2)} + RQ3 {len(KOLOM_RQ3)} = {out.shape[1]}")
print("\nRingkasan kelayakan:")
ringkas = pd.DataFrame({
    "semua": out.groupby("platform").size(),
    "common_is_usable": out.groupby("platform")["common_is_usable"].sum(),
    "rq1_eligible": out.groupby("platform")["rq1_eligible"].sum(),
    "rq2_eligible": out.groupby("platform")["rq2_eligible"].sum(),
    "rq2_eligible_text_only": out.groupby("platform")["rq2_eligible_text_only"].sum(),
    "rq3_eligible (A)": out.groupby("platform")["rq3_eligible"].sum(),
    "rq3_eligible_ext (A+B)": out.groupby("platform")["rq3_eligible_ext"].sum(),
})
ringkas.loc["total"] = ringkas.sum()
print(ringkas.to_string())
bulan = pd.DataFrame({
    "rq1_x": out[out["rq1_eligible"] & (out["platform"] == "x")].groupby("common_year_month").size(),
    "rq1_youtube": out[out["rq1_eligible"] & (out["platform"] == "youtube")].groupby("common_year_month").size(),
    "rq3_x": out[out["rq3_eligible"] & (out["platform"] == "x")].groupby("common_year_month").size(),
    "rq3_youtube": out[out["rq3_eligible"] & (out["platform"] == "youtube")].groupby("common_year_month").size(),
}).fillna(0).astype(int)
print("\nDokumen layak per bulan (cakupan waktu tidak merata, terutama YouTube):")
bulan.T

Kolom: asli 24 + umum 14 + RQ1 5 + RQ2 8 + RQ3 7 = 58

Ringkasan kelayakan:
            semua  common_is_usable  rq1_eligible  rq2_eligible  rq2_eligible_text_only  rq3_eligible (A)  rq3_eligible_ext (A+B)
platform                                                                                                                         
x           24535             24061         14216          3164                    2992             14508                   14592
youtube   1085353            976968         79417         45967                    4394             80428                  261093
total     1109888           1001029         93633         49131                    7386             94936                  275685

Dokumen layak per bulan (cakupan waktu tidak merata, terutama YouTube):


common_year_month,2025-01,2025-02,2025-03,2025-04,2025-05,2025-06,2025-07,2025-08,2025-09,2025-10,2025-11,2025-12,2026-01,2026-02,2026-03,2026-04,2026-05,2026-06,2026-07,2026-08,2026-09,2026-10
rq1_x,416,203,103,267,271,263,208,684,1055,1835,435,370,1234,760,1030,803,794,608,445,852,1580,0
rq1_youtube,2598,1220,134,669,1394,86,214,355,16427,6484,1024,1134,1546,4933,2925,2797,1498,6102,2636,6627,18399,215
rq3_x,432,213,108,274,287,271,230,705,1069,1869,450,371,1256,766,1054,817,818,614,454,861,1589,0
rq3_youtube,2545,1234,139,683,1466,63,229,369,16577,6097,988,1161,1533,5261,3106,2961,1576,6214,2704,6832,18464,226


Dataset akhir memiliki **58 kolom**: 24 kolom asli, 14 kolom `common_`, 5 kolom `rq1_`, 8 kolom `rq2_`, dan 7 kolom `rq3_`. Kolom bantu sementara (berawalan `_`) dibuang. Ringkasan kelayakan menunjukkan perbedaan skala antar-RQ: RQ1 memiliki 63.375 dokumen, RQ2 31.853 (5.632 jika hanya dari teks sendiri), dan RQ3 64.735 pada tingkat A (179.445 jika tingkat B disertakan). Pada X, hampir seluruh dokumen layak adalah dokumen relevan (14.216 untuk RQ1), sedangkan pada YouTube hanya 49.159 dari 644.848 dokumen; selebihnya adalah komentar yang tidak membahas keamanan pangan. Jumlah dokumen layak per bulan memperlihatkan ketimpangan waktu yang sama dengan bagian 2.1.

## 6.2. Validasi

In [26]:
bool_baru = [c for c in out.columns if c not in ORIG_COLS and out[c].dtype == bool]
cek = {
    "jumlah baris sama dengan input": len(out) == len(df) == (NROWS or meta["rows"]["total"]),
    "doc_id unik": bool(out["doc_id"].is_unique),
    "kolom asli tidak berubah (hash isi sama)": int(pd.util.hash_pandas_object(out[ORIG_COLS], index=False).sum() % (2**63)) == HASH_ASLI,
    "kolom flag baru tidak mengandung missing": all(out[c].notna().all() for c in bool_baru),
    "rq1_eligible subset relevan dan layak pakai": bool((out["rq1_eligible"] <= (out["is_relevant_candidate"] & out["common_is_usable"])).all()),
    "rq1_eligible punya rq1_text_light dan rq1_tokens": bool(out.loc[out["rq1_eligible"], ["rq1_text_light", "rq1_tokens"]].notna().all().all()),
    "rq1_eligible: minimal token terpenuhi": bool((out.loc[out["rq1_eligible"], "rq1_n_tokens"] >= MIN_RQ1_TOKENS).all()),
    "rq1_text_light tanpa URL, mention, atau emoji": not out["rq1_text_light"].str.contains(r"https?://|www\.|@\w|[\U0001F300-\U0001FAFF]", na=False).any(),
    "rq1_eligible tidak memuat spam atau duplikat teks": not (out["rq1_eligible"] & (out["common_is_spam"] | ~out["common_is_dup_first"])).any(),
    "rq2_eligible subset rq1_eligible dan punya provinsi": bool((out["rq2_eligible"] <= out["rq1_eligible"]).all() and out.loc[out["rq2_eligible"], "rq2_province"].notna().all()),
    "rq2_province baku (38 provinsi)": bool(out["rq2_province"].dropna().isin(PROVINSI_PULAU).all()),
    "rq2_province tunggal atau kosong (tidak ada ';')": not out["rq2_province"].str.contains(";", na=False).any(),
    "rq2_eligible_text_only subset rq2_eligible": bool((out["rq2_eligible_text_only"] <= out["rq2_eligible"]).all()),
    "rq3_eligible subset dari rq3_eligible_ext": bool((out["rq3_eligible"] <= out["rq3_eligible_ext"]).all()),
    "rq3_eligible_ext punya rq3_text": bool(out.loc[out["rq3_eligible_ext"], "rq3_text"].notna().all()),
    "rq3 tidak memuat video dan spam": not (out["rq3_eligible_ext"] & ((out["doc_type"] == "video") | out["common_is_spam"])).any(),
    "rq3_text hanya memuat mention berupa @user": not out["rq3_text"].str.replace("@user", "", regex=False).str.contains(r"@\w", na=False).any(),
    "rq3_text tidak memuat tautan asli": not out["rq3_text"].str.contains(r"https?://|www\.", na=False).any(),
    "rq3_label_sample subset rq3_eligible": bool((out["rq3_label_sample"] <= out["rq3_eligible"]).all()),
}
for nama, ok in cek.items():
    print("PASS" if ok else "FAIL", "-", nama)
assert all(cek.values()), "Validasi dataset processed gagal"

PASS - jumlah baris sama dengan input
PASS - doc_id unik
PASS - kolom asli tidak berubah (hash isi sama)
PASS - kolom flag baru tidak mengandung missing
PASS - rq1_eligible subset relevan dan layak pakai
PASS - rq1_eligible punya rq1_text_light dan rq1_tokens
PASS - rq1_eligible: minimal token terpenuhi
PASS - rq1_text_light tanpa URL, mention, atau emoji
PASS - rq1_eligible tidak memuat spam atau duplikat teks
PASS - rq2_eligible subset rq1_eligible dan punya provinsi
PASS - rq2_province baku (38 provinsi)
PASS - rq2_province tunggal atau kosong (tidak ada ';')
PASS - rq2_eligible_text_only subset rq2_eligible
PASS - rq3_eligible subset dari rq3_eligible_ext
PASS - rq3_eligible_ext punya rq3_text
PASS - rq3 tidak memuat video dan spam
PASS - rq3_text hanya memuat mention berupa @user
PASS - rq3_text tidak memuat tautan asli
PASS - rq3_label_sample subset rq3_eligible


Seluruh 19 pemeriksaan **lulus**, termasuk: jumlah baris sama dengan masukan, `doc_id` unik, **kolom asli tidak berubah** (hash isi sama), semua himpunan kelayakan konsisten (RQ2 subset RQ1, tingkat A subset A+B, sampel pelabelan subset tingkat A), tidak ada spam, video, atau duplikat teks yang lolos ke himpunan yang tidak semestinya, `rq1_text_light` tanpa URL/mention/emoji, `rq3_text` hanya memuat mention berupa `@user`, dan `rq2_province` selalu salah satu dari 38 provinsi baku.

## 6.3. Kamus Data, Penyimpanan, dan Log

In [27]:
KAMUS = [
    # (kolom, kelompok, deskripsi)
    ("doc_id", "asli", "ID dokumen gabungan: yt_v_<video>, yt_c_<komentar>, x_<tweet>."),
    ("platform", "asli", "Sumber data: youtube atau x."),
    ("doc_type", "asli", "Jenis dokumen: video, comment, reply (YouTube/X), post, quote (X)."),
    ("parent_doc_id", "asli", "Dokumen induk: komentar/video untuk reply, tweet yang dibalas atau dikutip."),
    ("context_id", "asli", "Konteks percakapan: video (YouTube) atau akar percakapan yang terbaca (X)."),
    ("source_name", "asli", "Nama saluran (YouTube); kosong untuk X."),
    ("source_category", "asli", "Kategori saluran (YouTube); kosong untuk X."),
    ("author_hash", "asli", "Pseudonim penulis; kosong untuk video."),
    ("text_raw", "asli", "Teks asli."),
    ("text_clean", "asli", "Teks setelah pembersihan umum (pre-processing tahap 1)."),
    ("published_at_utc", "asli", "Waktu terbit (UTC, ISO 8601)."),
    ("like_count", "asli", "Jumlah like."),
    ("reply_count", "asli", "Jumlah balasan; kosong untuk reply dan sebagian video YouTube."),
    ("url", "asli", "Tautan dokumen."),
    ("location_names", "asli", "Nama lokasi yang disebut (dipisah ';')."),
    ("province", "asli", "Provinsi dari lokasi yang disebut (dipisah ';')."),
    ("collected_at_utc", "asli", "Waktu pengumpulan data."),
    ("quoted_doc_id", "asli", "Tweet yang dikutip (khusus X)."),
    ("is_suspected_spam", "asli", "Dugaan spam (dari merge)."),
    ("is_repeated_text", "asli", "Teks persis sama dengan dokumen lain (dari merge)."),
    ("mentions_mbg", "asli", "Teks menyebut MBG/SPPG/BGN dan sejenisnya (dari merge)."),
    ("mentions_food_safety", "asli", "Teks memuat kata terkait keamanan/kebersihan pangan (dari merge)."),
    ("context_mentions_mbg", "asli", "Konteks (video) menyebut MBG (dari merge); hampir selalu True pada YouTube."),
    ("is_relevant_candidate", "asli", "mentions_food_safety DAN (mentions_mbg ATAU context_mentions_mbg) (dari merge)."),
    ("common_published_date_wib", "umum", "Tanggal terbit (WIB) YYYY-MM-DD."),
    ("common_year_month", "umum", "Bulan terbit (WIB) YYYY-MM."),
    ("common_in_overlap_period", "umum", "Terbit dalam periode tumpang tindih X dan YouTube."),
    ("common_text_len", "umum", "Panjang text_clean (karakter)."),
    ("common_n_words", "umum", "Jumlah kata alfabet setelah NFKC (tanpa URL dan mention)."),
    ("common_is_too_short", "umum", f"Kurang dari {MIN_WORDS} kata."),
    ("common_is_spam", "umum", "is_suspected_spam ATAU memuat merek situs judi yang berulang pada dokumen spam."),
    ("common_dup_group_size", "umum", "Jumlah dokumen dengan teks sama (setelah NFKC, tanpa URL/mention/tanda baca); bisa dipakai sebagai bobot."),
    ("common_is_dup_first", "umum", "Kemunculan pertama (menurut waktu) dalam grup teks yang sama."),
    ("common_is_dup_same_author", "umum", "Penulis yang sama mengulang teks yang sama setelah kemunculan pertamanya."),
    ("common_is_non_indonesian", "umum", "Dominan bahasa Inggris (>= 30% kata fungsi Inggris) atau beraksara non-Latin."),
    ("common_context_mentions_food_safety", "umum", "Konteks (video/unggahan akar) menyebut kata kunci keamanan pangan."),
    ("common_is_usable", "umum", "Bukan spam, cukup panjang, bahasa Indonesia, dan bukan pengulangan penulis yang sama."),
    ("common_audit_sample", "umum", "Sampel audit manual flag relevansi (100 per platform per nilai is_relevant_candidate)."),
    ("rq1_text_light", "RQ1", "Teks ringan untuk model embedding: huruf kecil, tanpa URL/mention/emoji, ejaan baku. Terisi hanya bila kandidat RQ1."),
    ("rq1_tokens", "RQ1", "Token untuk LDA/NMF: stopword dibuang dan di-stem (Sastrawi). Terisi hanya bila kandidat RQ1."),
    ("rq1_n_tokens", "RQ1", "Jumlah token pada rq1_tokens."),
    ("rq1_issue_categories", "RQ1", "Kategori isu yang kata kuncinya muncul (label lemah, dipisah ';')."),
    ("rq1_eligible", "RQ1", "Layak topic modelling: relevan, layak pakai, kemunculan pertama teks, dan >= token minimum."),
    ("rq2_province_list", "RQ2", "Provinsi yang disebut pada teks sendiri (baku, dipisah ';')."),
    ("rq2_n_provinces", "RQ2", "Jumlah provinsi yang disebut pada teks sendiri."),
    ("rq2_province_context", "RQ2", "Provinsi tunggal yang diwarisi dari konteks bila teks sendiri tidak menyebut provinsi."),
    ("rq2_location_source", "RQ2", "Sumber lokasi: text, context, atau none."),
    ("rq2_province", "RQ2", "Provinsi tunggal final (teks sendiri, atau konteks); kosong bila tidak ada atau lebih dari satu."),
    ("rq2_island_group", "RQ2", "Kelompok pulau dari rq2_province."),
    ("rq2_eligible", "RQ2", "rq1_eligible dan punya rq2_province."),
    ("rq2_eligible_text_only", "RQ2", "rq2_eligible dengan lokasi dari teks sendiri (tanpa pewarisan)."),
    ("rq3_tier", "RQ3", "A = menyebut keamanan pangan sendiri (utama); B = konteksnya membahas keamanan pangan (sensitivitas)."),
    ("rq3_eligible", "RQ3", "Tingkat A, bukan video, layak pakai."),
    ("rq3_eligible_ext", "RQ3", "Tingkat A atau B."),
    ("rq3_text", "RQ3", "Teks minimal untuk sentimen: huruf kecil, <URL>, @user, negasi dan emoji dipertahankan, tanpa stopword/stemming."),
    ("rq3_text_emoji_id", "RQ3", "rq3_text dengan emoji diganti namanya (bahasa Indonesia), untuk metode berbasis kamus kata."),
    ("rq3_n_emoji", "RQ3", "Jumlah emoji pada text_clean."),
    ("rq3_label_sample", "RQ3", f"Sampel pelabelan manual ({LABEL_SAMPLE_PER_PLATFORM} dokumen tingkat A per platform)."),
]
kamus = pd.DataFrame(KAMUS, columns=["kolom", "kelompok", "deskripsi"])
kamus["tipe"] = kamus["kolom"].map(out.dtypes.astype(str))
kamus["terisi"] = kamus["kolom"].map(lambda c: round(float(out[c].notna().mean()), 4))
assert set(kamus["kolom"]) == set(out.columns) and kamus["kolom"].is_unique, set(out.columns) ^ set(kamus["kolom"])
print("Kamus data mencakup seluruh", len(kamus), "kolom; kolom baru:", int((kamus["kelompok"] != "asli").sum()))
kamus[kamus["kelompok"] != "asli"].head(8)

Kamus data mencakup seluruh 58 kolom; kolom baru: 34


,kolom,kelompok,deskripsi,tipe,terisi
24,common_published_date_wib,umum,Tanggal terbit (WIB) YYYY-MM-DD.,str,1.0
25,common_year_month,umum,Bulan terbit (WIB) YYYY-MM.,str,1.0
26,common_in_overlap_period,umum,Terbit dalam periode tumpang tindih X dan YouTube.,bool,1.0
27,common_text_len,umum,Panjang text_clean (karakter).,int64,1.0
28,common_n_words,umum,Jumlah kata alfabet setelah NFKC (tanpa URL dan mention).,int32,1.0
29,common_is_too_short,umum,Kurang dari 3 kata.,bool,1.0
30,common_is_spam,umum,is_suspected_spam ATAU memuat merek situs judi yang berulang pada dokumen spam.,bool,1.0
31,common_dup_group_size,umum,"Jumlah dokumen dengan teks sama (setelah NFKC, tanpa URL/mention/tanda baca); bisa dipakai sebagai bobot.",int32,1.0


In [28]:
t0 = time.time()
out.to_csv(OUT_CSV, index=False, encoding="utf-8", lineterminator="\n")
kamus.to_csv(OUT_DICT, index=False, encoding="utf-8")
log_df = pd.DataFrame(log)
log_df.to_csv(OUT_LOG, index=False, encoding="utf-8")
print(f"Disimpan dalam {time.time() - t0:.0f} detik")
for berkas in [OUT_CSV, OUT_DICT, OUT_LOG]:
    print(f"  {berkas.relative_to(BASE)}: {berkas.stat().st_size / 1e6:.1f} MB")

cek_baca = pd.read_csv(OUT_CSV, dtype=str, keep_default_na=False, na_values=[""])
print("\nBaca ulang: dimensi sama =", cek_baca.shape == out.shape, "| doc_id utuh =", bool((cek_baca["doc_id"].values == out["doc_id"].values).all()))
for kolom in ["text_clean", "rq1_tokens", "rq3_text"]:
    sama = (cek_baca[kolom].fillna("").values == out[kolom].fillna("").values).all()
    print(f"  {kolom} utuh setelah dibaca ulang: {bool(sama)}")
for kolom in ["rq1_eligible", "rq2_eligible", "rq3_eligible", "rq3_label_sample"]:
    print(f"  {kolom}: {int((cek_baca[kolom] == 'True').sum())} True")
log_df

Disimpan dalam 19 detik
  data\processed\mbg_processed\mbg_processed.csv: 887.1 MB
  data\processed\mbg_processed\data_dictionary.csv: 0.0 MB
  data\processed\mbg_processed\preprocessing_log.csv: 0.0 MB

Baca ulang: dimensi sama = True | doc_id utuh = True
  text_clean utuh setelah dibaca ulang: True
  rq1_tokens utuh setelah dibaca ulang: True
  rq3_text utuh setelah dibaca ulang: True
  rq1_eligible: 93633 True
  rq2_eligible: 49131 True
  rq3_eligible: 94936 True
  rq3_label_sample: 800 True


,langkah,keterangan,n_baris,n_ditandai
0,1.2 Muat dataset gabungan,"mbg_merged.csv; tipe bool dan angka dikonversi, kolom asli tidak diubah isinya",1109888,NaN
1,2.1 Waktu,"common_published_date_wib, common_year_month, common_in_overlap_period",1109888,1106902.0
2,2.2 Normalisasi dasar dan panjang teks,"common_text_len, common_n_words, common_is_too_short",1109888,90028.0
3,2.3 Spam,common_is_spam = is_suspected_spam OR memuat merek situs judi yang berulang pada dokumen spam,1109888,2366.0
4,2.4 Teks berulang,"common_dup_group_size, common_is_dup_first, common_is_dup_same_author; tidak ada baris dibuang",1109888,19548.0
5,2.5 Bahasa,common_is_non_indonesian: dominan kata fungsi Inggris atau aksara non-Latin,1109888,1135.0
6,2.6 Konteks keamanan pangan,common_context_mentions_food_safety (video atau unggahan akar menyebut kata kunci keamanan pangan),1109888,262387.0
7,2.7 Dokumen layak pakai,"common_is_usable = bukan spam, cukup panjang, bahasa Indonesia, bukan pengulangan penulis yang sama",1109888,1001029.0
8,2.8 Sampel audit relevansi,common_audit_sample (100 dokumen per platform per nilai is_relevant_candidate),1109888,400.0
9,3.5 RQ1,"rq1_text_light, rq1_tokens, rq1_n_tokens, rq1_issue_categories, rq1_eligible",97491,93633.0


Kamus data (`data_dictionary.csv`) mendokumentasikan seluruh 58 kolom: kelompok, tipe, proporsi terisi, dan deskripsi. Dataset disimpan ke `data/processed/mbg_processed/mbg_processed.csv` (sekitar 548 MB, tidak di-commit ke Git) beserta kamus data dan log pre-processing. Pembacaan ulang menunjukkan berkas utuh: dimensi sama, `doc_id` dan kolom teks identik, serta hitungan kelayakan sesuai (63.375 untuk RQ1, 31.853 untuk RQ2, 64.735 untuk RQ3, dan 800 sampel pelabelan).

# 7. Ringkasan dan Catatan untuk Tahap Modelling

| Aspek | Hasil |
|---|---|
| Dataset | 1 berkas `mbg_processed.csv`: **669.383 baris**, **58 kolom** (24 asli + 34 baru); tidak ada baris dibuang |
| Dokumen layak pakai umum | 604.616 (90,3%) |
| RQ1 (topic modelling) | **63.375** dokumen layak; `rq1_text_light` (embedding) dan `rq1_tokens` (LDA/NMF); `rq1_issue_categories` sebagai label lemah |
| RQ2 (lokasi) | **31.853** dokumen layak, **5.632** di antaranya berlokasi dari teks sendiri; provinsi baku (38) dan kelompok pulau |
| RQ3 (sentimen) | **64.735** dokumen tingkat A (179.445 dengan tingkat B); `rq3_text` dan `rq3_text_emoji_id`; **800** sampel pelabelan |

**Hal yang perlu diperhatikan saat modelling**

1. **YouTube mendominasi (96% baris).** Sebagian besar hasil gabungan akan mewakili YouTube. Analisis sebaiknya dipisah atau dibobot per platform, dan perbedaan karakter teks (komentar pendek vs unggahan X vs deskripsi video) dipertimbangkan, terutama untuk topic modelling.
2. **Cakupan waktu tidak merata** (bagian 2.1 dan 6.1), dan koleksi YouTube belum final. Hindari membaca volume bulanan sebagai tren.
3. **Relevansi berbasis kata kunci.** `is_relevant_candidate` belum diukur presisinya; pelabelan manual sampel audit (`common_audit_sample`, 400 dokumen) perlu dilakukan. Kata kunci "keracunan" mendominasi (70% dokumen RQ1).
4. **RQ2 bergantung pada pewarisan lokasi** (82% dokumen layak) dan terkonsentrasi pada sedikit video. Laporkan hasil dengan dan tanpa pewarisan, gunakan jumlah video unik, dan gabungkan provinsi berdokumen sedikit per pulau. RQ2 juga memerlukan label topik dari RQ1.
5. **RQ3:** tingkat B sangat berisik dan hanya untuk uji sensitivitas. Pelabelan manual diperlukan untuk memvalidasi model sentimen dan memisahkan sentimen negatif umum dari kekhawatiran keamanan pangan.
6. **Keterbatasan teknik:** kamus ejaan tidak baku hanya 147 entri, stemming Sastrawi menghasilkan akar yang kadang tidak natural, dan deteksi bahasa hanya menyaring bahasa Inggris dan aksara non-Latin.
7. **Privasi dan Git.** Berkas data besar di-ignore oleh Git. `text_raw` dan `text_clean` masih dapat memuat `@mention`; bagikan data lewat penyimpanan terkontrol, bukan repository publik.
8. **Belum dikerjakan (sesuai lingkup TK2):** pemilihan dan pelatihan model topik, analisis lokasi antar-wilayah, klasifikasi sentimen, serta pelabelan manual.